# 15-minute recognition training

Upload **this notebook** to Google Colab and run the cells from top to bottom.
The training code is in the notebook. A GitHub clone is not required.

The data cell loads BTCUSD 5-minute candles from the public Delta Exchange
history API and writes `/content/btcusd_5m_raw.parquet`. It does not connect
to Google Drive and it does not ask for a file upload.

Use a GPU runtime. The fit predicts geometric candle families and market state
from 48 completed 15-minute bars. Each bar also carries causal support and
resistance distance, proximity, touch counts, and range position. After the
fit, the next cells draw the
learning curve, validation against the sealed test, the Optuna trial scores,
the label correlation heatmaps, and the SHAP ranking. Leave the checkpoint
in the Colab output folder. It is a research artifact.


## Setup

In [ ]:
import importlib.util
import subprocess
import sys

def _pip(*packages: str) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

if importlib.util.find_spec("torch") is None:
    _pip("torch")
_pip(
    "optuna", "shap", "xgboost", "scipy", "pandas", "pyarrow",
    "matplotlib", "structlog",
)
print("recognition dependencies ready")


## Refused column names

In [ ]:
# Refused column names, copied when this notebook is generated.
# The cell does not import the rest of the repository.
PATTERN_COLUMNS = (
    'doji',
    'hammer',
    'shooting_star',
    'bullish_engulfing',
    'bearish_engulfing',
    'inside_bar',
    'close_broke_high_24',
    'close_broke_low_24',
    'consecutive_closes',
)
NATIVE_STRUCTURE_COLS = (
    'hh_count',
    'hl_count',
    'lh_count',
    'll_count',
    'structure_bias',
    'last_swing_dir',
    'bars_since_swing',
    'swing_amp_atr',
    'unconfirmed_ext_atr',
    'trend_efficiency',
    'displacement_atr',
    'pct_with_trend',
    'price_vs_ema9_atr',
    'price_vs_ema21_atr',
    'price_vs_ema50_atr',
    'price_vs_ema200_atr',
    'ema9_vs_21_atr',
    'ema21_vs_50_atr',
    'ema50_vs_200_atr',
    'ema21_slope_atr',
    'ema50_slope_atr',
    'dist_to_support_atr',
    'dist_to_resistance_atr',
    'support_touch_count',
    'resistance_touch_count',
    'range_width_atr',
    'range_width_pctile',
    'atr_contraction',
    'breakout_size_atr',
    'breakout_vol_ratio',
    'pre_breakout_comp',
    'bars_since_breakout',
    'retest_dist_atr',
    'failed_break',
    'peak_diff_atr',
    'trough_diff_atr',
    'peak_sep_bars',
    'trough_sep_bars',
    'dist_neck_atr',
    'high_slope_atr',
    'low_slope_atr',
    'convergence',
    'width_now_atr',
    'pole_disp_atr',
    'flag_width_atr',
    'flag_slope_atr',
)


## Contract

In [ ]:
"""Frozen constants for the 15-minute recognition research track.

The live forecast path is a different package. Nothing here is an order.
"""

from __future__ import annotations


SPEC_VERSION = "recognition-15m-v2"
INSTRUMENT = "BTCUSD"
SOURCE_MINUTES = 5
BAR_MINUTES = 15
WINDOW_BARS = 48
PURGE_BARS = 48
ATR_PERIOD = 14
VOLUME_BASELINE = 48
TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.15
TRAIN_STRIDE = 4
EVAL_STRIDE = 48
TCN_EPOCHS = 6
TCN_BATCH = 512
TCN_HIDDEN = 32
TCN_KERNEL = 3
TCN_DILATIONS = (1, 2, 4)
TCN_LEARNING_RATE = 1e-2
SEARCH_TRIALS = 30
SEARCH_EPOCHS = 20
SEARCH_PATIENCE = 4
FINAL_EPOCHS = 40
FINAL_PATIENCE = 6
DEVELOPMENT_FOLDS = 3
CORRELATION_FLAG = 0.5
SHAP_SAMPLE = 2000
HIDDEN_CHOICES = (32, 64, 128)
BATCH_CHOICES = (256, 512)
CLIP_CHOICES = (5.0, 10.0, 20.0, 40.0)
COLAB_OUTPUT_RELATIVE = "export/recognition_15m/colab/report.json"
XGB_ESTIMATORS = 40
XGB_MAX_DEPTH = 3
SEED = 7
POS_WEIGHT_CLIP = 20.0
ZZ_TAU = 1.5
ZZ_EPS_ATR = 0.15
PRIOR_TREND_BARS = 5
TREND_FLAT_BAND = 0.001

DOJI_BODY_MAX = 0.10
WICK_BODY_MULTIPLE = 2.0
WICK_OPPOSITE_MAX = 0.20
WICK_BODY_MIN = 0.05
RANGE_ATR_MIN = 0.50
MARUBOZU_BODY_MIN = 0.85
MARUBOZU_WICK_MAX = 0.08
ENGULF_BODY_ATR_MIN = 0.25
HARAMI_PRIOR_ATR_MIN = 0.40
TWEEZER_BODY_ATR_MIN = 0.20
TWEEZER_ATR_TOL = 0.15
STAR_LARGE_BODY = 0.55
STAR_SMALL_BODY = 0.30
SOLDIER_BODY_MIN = 0.50
SOLDIER_WICK_MAX = 0.25

SOURCE_RELATIVE = "export/fusion_diagnostics/btcusd_5m_raw.parquet"
OUTPUT_RELATIVE = "export/recognition_15m/metrics.json"

RAW_CHANNELS = (
    "open_atr",
    "high_atr",
    "low_atr",
    "close_atr",
    "volume_ratio",
)
ENGINEERED_CHANNELS = (
    "body_size_atr",
    "upper_wick_atr",
    "lower_wick_atr",
    "range_atr",
    "body_to_range",
    "upper_wick_to_range",
    "lower_wick_to_range",
    "close_location",
    "body_sign",
    "close_change_atr",
    "volume_ratio",
    "level_support_dist_atr",
    "level_resistance_dist_atr",
    "level_at_support",
    "level_at_resistance",
    "level_support_touches",
    "level_resistance_touches",
    "level_range_position",
)
ALLOWED_CHANNELS = frozenset(RAW_CHANNELS) | frozenset(ENGINEERED_CHANNELS)
LEVEL_LOOKBACK = 60
LEVEL_NEAR_ATR = 1.0
LEVEL_TOUCH_ATR = 0.5

GEOMETRIC_CLASSES = (
    "long_lower_wick",
    "long_upper_wick",
    "doji",
    "marubozu",
    "engulfing",
    "harami",
    "piercing_dark_cloud",
    "tweezer",
    "star",
    "soldiers_crows",
)
MARKET_STATES = (
    "uptrend",
    "downtrend",
    "range_bound",
    "transition",
    "indeterminate",
)
STRUCTURE_TAGS = ("hh_hl", "lh_ll", "mixed", "none")

RULE_CARD_COLUMNS = (
    "bar_time",
    "bar_index",
    "geometric_class",
    "conventional_name",
    "directional_bias",
    "context",
    "abstain",
)

FORBIDDEN_PREFIXES = ("cdl_", "chp_", "sr_", "tl_", "bo_")
FORECAST_LABEL_COLUMNS = (
    "y_direction",
    "forward_return_24",
    "forward_log_return_24",
    "long_path_code",
    "short_path_code",
    "long_tp_first_24",
    "short_tp_first_24",
    "tp_pct",
    "sl_pct",
    "neutral_band",
    "buy",
    "sell",
    "hold",
    "expectancy",
    "sharpe",
    "win_rate",
)
FORBIDDEN_EXACT = frozenset(
    tuple(PATTERN_COLUMNS)
    + tuple(NATIVE_STRUCTURE_COLS)
    + FORECAST_LABEL_COLUMNS
    + RULE_CARD_COLUMNS
    + GEOMETRIC_CLASSES
    + MARKET_STATES
    + STRUCTURE_TAGS
)

HEADS_NOT_TRAINED = ("B", "C", "D", "F", "G", "H")
RECOGNITION_BUNDLE_DIR_NAME = "JackSparrow_Recognition_BTCUSD_15m"
RECOGNITION_DECISION_PATH = "recognition_15m"
RECOGNITION_CHECKPOINT_NAME = "tcn_colab.pt"

__all__ = [
    "ALLOWED_CHANNELS",
    "ATR_PERIOD",
    "BAR_MINUTES",
    "ENGINEERED_CHANNELS",
    "LEVEL_LOOKBACK",
    "LEVEL_NEAR_ATR",
    "LEVEL_TOUCH_ATR",
    "EVAL_STRIDE",
    "FORBIDDEN_EXACT",
    "FORBIDDEN_PREFIXES",
    "GEOMETRIC_CLASSES",
    "HEADS_NOT_TRAINED",
    "RECOGNITION_BUNDLE_DIR_NAME",
    "RECOGNITION_CHECKPOINT_NAME",
    "RECOGNITION_DECISION_PATH",
    "MARKET_STATES",
    "OUTPUT_RELATIVE",
    "POS_WEIGHT_CLIP",
    "PURGE_BARS",
    "RAW_CHANNELS",
    "RULE_CARD_COLUMNS",
    "SEED",
    "SOURCE_RELATIVE",
    "SPEC_VERSION",
    "STRUCTURE_TAGS",
    "TCN_BATCH",
    "TCN_DILATIONS",
    "TCN_EPOCHS",
    "TCN_HIDDEN",
    "TCN_KERNEL",
    "BATCH_CHOICES",
    "CLIP_CHOICES",
    "COLAB_OUTPUT_RELATIVE",
    "CORRELATION_FLAG",
    "DEVELOPMENT_FOLDS",
    "FINAL_EPOCHS",
    "FINAL_PATIENCE",
    "HIDDEN_CHOICES",
    "SEARCH_EPOCHS",
    "SEARCH_PATIENCE",
    "SEARCH_TRIALS",
    "SHAP_SAMPLE",
    "TCN_LEARNING_RATE",
    "TRAIN_FRACTION",
    "TRAIN_STRIDE",
    "VAL_FRACTION",
    "VOLUME_BASELINE",
    "WINDOW_BARS",
    "XGB_ESTIMATORS",
    "XGB_MAX_DEPTH",
]


## Completed 15-minute bars

In [ ]:
"""Resample completed 5-minute BTCUSD candles into 15-minute bars.

A bucket is emitted only when it contains the three clock-aligned 5-minute
bars. The next bucket is never consulted.
"""

from __future__ import annotations

from typing import Iterable

import numpy as np
import pandas as pd


_OHLCV = ("open", "high", "low", "close", "volume")
_BUCKET = pd.Timedelta(minutes=BAR_MINUTES)
_STEP = pd.Timedelta(minutes=SOURCE_MINUTES)


def _as_utc(values: pd.Series) -> pd.Series:
    """Parse timestamps as UTC.

    Args:
        values: Datetime-like values. Naive values are treated as UTC.

    Returns:
        Timezone-aware UTC timestamps.
    """
    parsed = pd.to_datetime(values, utc=True)
    return pd.Series(parsed, index=values.index)


def dedupe_timestamps(frame: pd.DataFrame) -> pd.DataFrame:
    """Keep the last row for each timestamp.

    Args:
        frame: Candle rows with a ``time`` column.

    Returns:
        Frame sorted by time, with one row per timestamp.
    """
    work = frame.copy()
    work["time"] = _as_utc(work["time"])
    work = work.sort_values("time", kind="mergesort")
    return work.drop_duplicates("time", keep="last").reset_index(drop=True)


def resample_completed_15m(frame: pd.DataFrame) -> pd.DataFrame:
    """Aggregate contiguous 5-minute bars into completed 15-minute bars.

    Args:
        frame: Columns ``time``, ``open``, ``high``, ``low``, ``close``,
            and ``volume``.

    Returns:
        One row per complete 15-minute bucket, sorted by bucket open time.

    Raises:
        ValueError: If a required column is missing.
    """
    required = {"time", *_OHLCV}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"Candle frame is missing columns: {sorted(missing)}")

    work = frame.loc[:, ["time", *_OHLCV]].copy()
    work["time"] = _as_utc(work["time"])
    work = work.sort_values("time", kind="mergesort")
    bucket = work["time"].dt.floor(_BUCKET)
    grouped = work.groupby(bucket, sort=True)
    rows: list[dict[str, object]] = []
    for start, group in grouped:
        if not _bucket_is_complete(group["time"].tolist(), start):
            continue
        ordered = group.sort_values("time", kind="mergesort")
        rows.append(
            {
                "time": pd.Timestamp(start),
                "open": float(ordered["open"].iloc[0]),
                "high": float(ordered["high"].max()),
                "low": float(ordered["low"].min()),
                "close": float(ordered["close"].iloc[-1]),
                "volume": float(ordered["volume"].sum()),
            }
        )
    if not rows:
        return pd.DataFrame(columns=["time", *_OHLCV])
    out = pd.DataFrame(rows)
    out["time"] = pd.to_datetime(out["time"], utc=True)
    return out.reset_index(drop=True)


def _bucket_is_complete(stamps: Iterable[pd.Timestamp], start: pd.Timestamp) -> bool:
    """Return True when the bucket holds exactly the three expected stamps."""
    expected = {
        pd.Timestamp(start),
        pd.Timestamp(start) + _STEP,
        pd.Timestamp(start) + 2 * _STEP,
    }
    seen = [pd.Timestamp(stamp) for stamp in stamps]
    if len(seen) != 3 or len(set(seen)) != 3:
        return False
    return set(seen) == expected


def synthetic_5m(
    buckets: int,
    *,
    start: str = "2024-01-01T00:00:00Z",
    price: float = 100.0,
) -> pd.DataFrame:
    """Build a gap-free 5-minute frame covering ``buckets`` complete groups.

    Args:
        buckets: Number of 15-minute groups.
        start: UTC start, aligned to a 15-minute boundary by the caller.
        price: Flat price level.

    Returns:
        OHLCV frame with three 5-minute rows per bucket.
    """
    count = buckets * 3
    stamps = pd.date_range(start, periods=count, freq="5min", tz="UTC")
    level = np.full(count, price, dtype=np.float64)
    return pd.DataFrame(
        {
            "time": stamps,
            "open": level,
            "high": level + 1.0,
            "low": level - 1.0,
            "close": level + 0.25,
            "volume": np.full(count, 10.0),
        }
    )


## Causal features

In [ ]:
"""Causal candle geometry for the recognition tensor.

Every value at bar ``t`` uses bars at or before ``t``. Pattern-rule columns
are not computed here.
"""

from __future__ import annotations

import numpy as np
import pandas as pd


_EPS = 1e-9


def causal_atr(high: np.ndarray, low: np.ndarray, close: np.ndarray, period: int) -> np.ndarray:
    """Simple moving average of true range, using the previous close.

    Args:
        high: High prices.
        low: Low prices.
        close: Close prices.
        period: Averaging window. Earlier bars stay NaN.

    Returns:
        ATR aligned to ``close``. Bar ``t`` uses true ranges through ``t``.
    """
    n = len(close)
    prev = np.empty(n, dtype=np.float64)
    prev[0] = close[0]
    prev[1:] = close[:-1]
    true_range = np.maximum(
        high - low,
        np.maximum(np.abs(high - prev), np.abs(low - prev)),
    )
    atr = np.full(n, np.nan, dtype=np.float64)
    if n < period:
        return atr
    window = np.ones(period, dtype=np.float64) / float(period)
    valid = np.convolve(true_range, window, mode="valid")
    atr[period - 1 :] = valid
    return atr


def volume_baseline(volume: np.ndarray, window: int) -> np.ndarray:
    """Trailing mean of volume ending at the current bar.

    Args:
        volume: Volume per bar.
        window: Lookback length, including the current bar.

    Returns:
        Baseline aligned to ``volume``. Earlier bars stay NaN.
    """
    n = len(volume)
    baseline = np.full(n, np.nan, dtype=np.float64)
    if n < window:
        return baseline
    kernel = np.ones(window, dtype=np.float64) / float(window)
    valid = np.convolve(volume.astype(np.float64), kernel, mode="valid")
    baseline[window - 1 :] = valid
    return baseline


def add_engineered_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Append scale-normalized geometry columns.

    Args:
        frame: Completed 15-minute OHLCV sorted by time.

    Returns:
        Copy with ``ENGINEERED_CHANNELS`` plus ``atr`` and ``volume_baseline``.
        ``atr`` and ``volume_baseline`` are scales, not model channels.

    Raises:
        ValueError: If a price column is missing.
    """
    required = {"open", "high", "low", "close", "volume"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"Feature frame is missing columns: {sorted(missing)}")

    out = frame.copy()
    open_ = out["open"].to_numpy(dtype=np.float64)
    high = out["high"].to_numpy(dtype=np.float64)
    low = out["low"].to_numpy(dtype=np.float64)
    close = out["close"].to_numpy(dtype=np.float64)
    volume = out["volume"].to_numpy(dtype=np.float64)
    atr = causal_atr(high, low, close, ATR_PERIOD)
    baseline = volume_baseline(volume, VOLUME_BASELINE)
    span = high - low
    safe_span = span > 0.0
    body = np.abs(close - open_)
    upper = high - np.maximum(open_, close)
    lower = np.minimum(open_, close) - low
    atr_ready = np.isfinite(atr)
    safe_atr = np.where(atr_ready, np.maximum(atr, _EPS), np.nan)
    base_ready = np.isfinite(baseline)
    safe_base = np.where(base_ready, np.maximum(baseline, _EPS), np.nan)
    zeros = np.zeros(len(close), dtype=np.float64)
    midpoint = np.full(len(close), 0.5, dtype=np.float64)

    out["atr"] = atr
    out["volume_baseline"] = baseline
    out["body_size_atr"] = body / safe_atr
    out["upper_wick_atr"] = upper / safe_atr
    out["lower_wick_atr"] = lower / safe_atr
    out["range_atr"] = span / safe_atr
    out["body_to_range"] = np.divide(body, span, out=zeros.copy(), where=safe_span)
    out["upper_wick_to_range"] = np.divide(upper, span, out=zeros.copy(), where=safe_span)
    out["lower_wick_to_range"] = np.divide(lower, span, out=zeros.copy(), where=safe_span)
    out["close_location"] = np.divide(close - low, span, out=midpoint, where=safe_span)
    out["body_sign"] = np.sign(close - open_)
    delta = np.empty(len(close), dtype=np.float64)
    delta[0] = np.nan
    delta[1:] = close[1:] - close[:-1]
    out["close_change_atr"] = delta / safe_atr
    out["volume_ratio"] = volume / safe_base
    levels = causal_level_channels(high, low, close, atr)
    for name, values in levels.items():
        out[name] = values
    for name in ENGINEERED_CHANNELS:
        if name not in out.columns:
            raise ValueError(f"Engineered channel was not computed: {name}")
    return out


def causal_level_channels(
    high: np.ndarray,
    low: np.ndarray,
    close: np.ndarray,
    atr: np.ndarray,
) -> dict[str, np.ndarray]:
    """Support and resistance from swings confirmed by a 1.5 ATR reversal.

    A swing high or low is known at the bar where price has reversed by
    ``ZZ_TAU`` ATR. Bars after that confirmation are not read. The active
    level is the nearest confirmed extreme inside ``LEVEL_LOOKBACK`` bars.
    When that window has no extreme on one side, the last confirmed extreme
    on that side remains the level. Rows stay NaN until both sides exist.

    Args:
        high: High prices.
        low: Low prices.
        close: Close prices.
        atr: Causal ATR aligned to ``close``.

    Returns:
        The seven ``level_*`` engineered channels.
    """
    n = len(close)
    support_dist = np.full(n, np.nan, dtype=np.float64)
    resistance_dist = np.full(n, np.nan, dtype=np.float64)
    at_support = np.full(n, np.nan, dtype=np.float64)
    at_resistance = np.full(n, np.nan, dtype=np.float64)
    support_touches = np.full(n, np.nan, dtype=np.float64)
    resistance_touches = np.full(n, np.nan, dtype=np.float64)
    range_position = np.full(n, np.nan, dtype=np.float64)
    if n == 0:
        return _level_frame(
            support_dist,
            resistance_dist,
            at_support,
            at_resistance,
            support_touches,
            resistance_touches,
            range_position,
        )

    highs: list[tuple[int, float]] = []
    lows: list[tuple[int, float]] = []
    high_cursor = 0
    low_cursor = 0
    direction = 1
    extreme_price = float(high[0])
    for index in range(n):
        atr_value = _positive_atr(atr[index], float(close[index]))
        threshold = ZZ_TAU * atr_value
        if direction == 1:
            if float(high[index]) >= extreme_price:
                extreme_price = float(high[index])
            elif (extreme_price - float(low[index])) >= threshold:
                highs.append((index, extreme_price))
                direction = -1
                extreme_price = float(low[index])
        elif float(low[index]) <= extreme_price:
            extreme_price = float(low[index])
        elif (float(high[index]) - extreme_price) >= threshold:
            lows.append((index, extreme_price))
            direction = 1
            extreme_price = float(high[index])

        lookback_start = index - LEVEL_LOOKBACK + 1
        high_cursor = _advance_cursor(highs, high_cursor, lookback_start)
        low_cursor = _advance_cursor(lows, low_cursor, lookback_start)
        resistance = _select_level(highs, high_cursor, float(close[index]), below=False)
        support = _select_level(lows, low_cursor, float(close[index]), below=True)
        if support is None or resistance is None:
            continue
        support_dist[index] = (float(close[index]) - support) / atr_value
        resistance_dist[index] = (resistance - float(close[index])) / atr_value
        at_support[index] = (
            1.0 if abs(float(close[index]) - support) < LEVEL_NEAR_ATR * atr_value else 0.0
        )
        at_resistance[index] = (
            1.0
            if abs(float(close[index]) - resistance) < LEVEL_NEAR_ATR * atr_value
            else 0.0
        )
        support_touches[index] = _touch_count(
            lows, low_cursor, support, LEVEL_TOUCH_ATR * atr_value
        )
        resistance_touches[index] = _touch_count(
            highs, high_cursor, resistance, LEVEL_TOUCH_ATR * atr_value
        )
        span = resistance - support
        if span > 0.0:
            range_position[index] = (float(close[index]) - support) / span
        else:
            range_position[index] = 0.5
    return _level_frame(
        support_dist,
        resistance_dist,
        at_support,
        at_resistance,
        support_touches,
        resistance_touches,
        range_position,
    )


def _positive_atr(atr_value: float, price: float) -> float:
    """ATR divisor. A non-positive reading falls back to one percent of price."""
    if np.isfinite(atr_value) and float(atr_value) > 0.0:
        return float(atr_value)
    return max(abs(price) * 0.01, _EPS)


def _advance_cursor(points: list[tuple[int, float]], cursor: int, start: int) -> int:
    """Drop confirmed swings older than the lookback."""
    while cursor < len(points) and points[cursor][0] < start:
        cursor += 1
    return cursor


def _select_level(
    points: list[tuple[int, float]],
    cursor: int,
    price: float,
    *,
    below: bool,
) -> float | None:
    """Nearest extreme on one side of price, else the closest, else the last."""
    if not points:
        return None
    active = points[cursor:]
    pool = active if active else points[-1:]
    chosen: float | None = None
    for _, level in pool:
        if below and level <= price and (chosen is None or level > chosen):
            chosen = level
        if not below and level >= price and (chosen is None or level < chosen):
            chosen = level
    if chosen is not None:
        return chosen
    return min(pool, key=lambda item: abs(item[1] - price))[1]


def _touch_count(
    points: list[tuple[int, float]],
    cursor: int,
    level: float,
    band: float,
) -> float:
    """How many lookback extremes sit inside ``band`` of ``level``."""
    return float(sum(1 for _, price in points[cursor:] if abs(price - level) <= band))


def _level_frame(
    support_dist: np.ndarray,
    resistance_dist: np.ndarray,
    at_support: np.ndarray,
    at_resistance: np.ndarray,
    support_touches: np.ndarray,
    resistance_touches: np.ndarray,
    range_position: np.ndarray,
) -> dict[str, np.ndarray]:
    """Map level arrays onto the engineered channel names."""
    return {
        "level_support_dist_atr": support_dist,
        "level_resistance_dist_atr": resistance_dist,
        "level_at_support": at_support,
        "level_at_resistance": at_resistance,
        "level_support_touches": support_touches,
        "level_resistance_touches": resistance_touches,
        "level_range_position": range_position,
    }


def first_finite_bar(features: pd.DataFrame) -> int:
    """Return the first bar index whose engineered channels are all finite.

    Args:
        features: Frame returned by :func:`add_engineered_features`.

    Returns:
        Bar index. ``len(features)`` when no bar is finite.
    """
    matrix = features.loc[:, list(ENGINEERED_CHANNELS)].to_numpy(dtype=np.float64)
    finite = np.isfinite(matrix).all(axis=1)
    hits = np.flatnonzero(finite)
    if len(hits) == 0:
        return len(features)
    return int(hits[0])


def safe_scale(value: float) -> float:
    """Return a positive divisor for a price or volume scale."""
    if not np.isfinite(value) or value <= 0.0:
        return _EPS
    return float(value)


## Geometric labels

In [ ]:
"""Geometric labels and the causal zigzag market state.

Labels at bar ``t`` use bars at or before ``t``. Conventional names and
directional bias are written only onto the rule card.
"""

from __future__ import annotations

from typing import Iterable

import numpy as np
import pandas as pd


_STATE_INDEX = {name: index for index, name in enumerate(MARKET_STATES)}
_TAG_INDEX = {name: index for index, name in enumerate(STRUCTURE_TAGS)}


def _parts(
    open_: np.ndarray,
    high: np.ndarray,
    low: np.ndarray,
    close: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Return body, upper wick, lower wick, range, body ratio, and body sign."""
    span = high - low
    body = np.abs(close - open_)
    upper = high - np.maximum(open_, close)
    lower = np.minimum(open_, close) - low
    ratio = np.divide(body, span, out=np.zeros(len(span)), where=span > 0.0)
    sign = np.sign(close - open_)
    return body, upper, lower, span, ratio, sign


def geometry_matrix(
    open_: np.ndarray,
    high: np.ndarray,
    low: np.ndarray,
    close: np.ndarray,
    atr: np.ndarray,
) -> np.ndarray:
    """Multi-label geometric families aligned to each bar.

    Args:
        open_: Open prices.
        high: High prices.
        low: Low prices.
        close: Close prices.
        atr: Causal ATR. Non-finite rows stay negative.

    Returns:
        Float array of shape ``(n, len(GEOMETRIC_CLASSES))`` with 0/1 values.
    """
    n = len(close)
    body, upper, lower, span, body_ratio, sign = _parts(open_, high, low, close)
    upper_ratio = np.divide(upper, span, out=np.zeros(n), where=span > 0.0)
    lower_ratio = np.divide(lower, span, out=np.zeros(n), where=span > 0.0)
    atr_ok = np.isfinite(atr) & (atr > 0.0)
    wide = atr_ok & (span > RANGE_ATR_MIN * atr)

    long_lower = (
        wide
        & (lower >= WICK_BODY_MULTIPLE * body)
        & (upper_ratio < WICK_OPPOSITE_MAX)
        & (body_ratio > WICK_BODY_MIN)
    )
    long_upper = (
        wide
        & (upper >= WICK_BODY_MULTIPLE * body)
        & (lower_ratio < WICK_OPPOSITE_MAX)
        & (body_ratio > WICK_BODY_MIN)
    )
    doji = wide & (body_ratio < DOJI_BODY_MAX)
    marubozu = (
        wide
        & (body_ratio >= MARUBOZU_BODY_MIN)
        & (upper_ratio < MARUBOZU_WICK_MAX)
        & (lower_ratio < MARUBOZU_WICK_MAX)
    )

    labels = np.zeros((n, len(GEOMETRIC_CLASSES)), dtype=np.float32)
    labels[:, 0] = long_lower
    labels[:, 1] = long_upper
    labels[:, 2] = doji
    labels[:, 3] = marubozu
    if n >= 2:
        labels[1:, 4] = _engulfing(body, sign, atr, open_, close)
        labels[1:, 5] = _harami(body, sign, atr, open_, close)
        labels[1:, 6] = _piercing(sign, open_, close)
        labels[1:, 7] = _tweezer(high, low, body, sign, atr)
    if n >= 3:
        labels[2:, 8] = _star(body_ratio, sign, open_, close)
        labels[2:, 9] = _soldiers(body_ratio, upper_ratio, lower_ratio, sign, close)
    return labels


def _engulfing(
    body: np.ndarray,
    sign: np.ndarray,
    atr: np.ndarray,
    open_: np.ndarray,
    close: np.ndarray,
) -> np.ndarray:
    """Engulfing bits for bars 1..n-1."""
    prior_lo = np.minimum(open_[:-1], close[:-1])
    prior_hi = np.maximum(open_[:-1], close[:-1])
    cur_lo = np.minimum(open_[1:], close[1:])
    cur_hi = np.maximum(open_[1:], close[1:])
    large = (body[1:] > ENGULF_BODY_ATR_MIN * atr[1:]) & (
        body[:-1] > ENGULF_BODY_ATR_MIN * atr[1:]
    )
    opposite = sign[1:] * sign[:-1] < 0.0
    contains = (cur_lo <= prior_lo) & (cur_hi >= prior_hi)
    return (large & opposite & contains).astype(np.float32)


def _harami(
    body: np.ndarray,
    sign: np.ndarray,
    atr: np.ndarray,
    open_: np.ndarray,
    close: np.ndarray,
) -> np.ndarray:
    """Harami bits for bars 1..n-1."""
    prior_lo = np.minimum(open_[:-1], close[:-1])
    prior_hi = np.maximum(open_[:-1], close[:-1])
    cur_lo = np.minimum(open_[1:], close[1:])
    cur_hi = np.maximum(open_[1:], close[1:])
    prior_large = body[:-1] > HARAMI_PRIOR_ATR_MIN * atr[1:]
    inside = (cur_lo >= prior_lo) & (cur_hi <= prior_hi) & (body[1:] < body[:-1])
    opposite = sign[1:] * sign[:-1] < 0.0
    return (prior_large & inside & opposite & (body[1:] > 0.0)).astype(np.float32)


def _piercing(sign: np.ndarray, open_: np.ndarray, close: np.ndarray) -> np.ndarray:
    """Piercing or dark-cloud bits for bars 1..n-1."""
    midpoint = (open_[:-1] + close[:-1]) * 0.5
    piercing = (
        (sign[:-1] < 0.0)
        & (sign[1:] > 0.0)
        & (open_[1:] <= close[:-1])
        & (close[1:] >= midpoint)
        & (close[1:] < open_[:-1])
    )
    dark = (
        (sign[:-1] > 0.0)
        & (sign[1:] < 0.0)
        & (open_[1:] >= close[:-1])
        & (close[1:] <= midpoint)
        & (close[1:] > open_[:-1])
    )
    return (piercing | dark).astype(np.float32)


def _tweezer(
    high: np.ndarray,
    low: np.ndarray,
    body: np.ndarray,
    sign: np.ndarray,
    atr: np.ndarray,
) -> np.ndarray:
    """Tweezer bits for bars 1..n-1."""
    tol = TWEEZER_ATR_TOL * atr[1:]
    bodies = (body[1:] > TWEEZER_BODY_ATR_MIN * atr[1:]) & (
        body[:-1] > TWEEZER_BODY_ATR_MIN * atr[1:]
    )
    opposite = sign[1:] * sign[:-1] < 0.0
    tops = np.abs(high[1:] - high[:-1]) <= tol
    bottoms = np.abs(low[1:] - low[:-1]) <= tol
    return (bodies & opposite & (tops | bottoms)).astype(np.float32)


def _star(
    body_ratio: np.ndarray,
    sign: np.ndarray,
    open_: np.ndarray,
    close: np.ndarray,
) -> np.ndarray:
    """Star bits for bars 2..n-1."""
    first_large = body_ratio[:-2] >= STAR_LARGE_BODY
    middle_small = body_ratio[1:-1] <= STAR_SMALL_BODY
    last_large = body_ratio[2:] >= STAR_LARGE_BODY
    opposite = sign[2:] * sign[:-2] < 0.0
    midpoint = (open_[:-2] + close[:-2]) * 0.5
    passed = ((sign[2:] > 0.0) & (close[2:] >= midpoint)) | (
        (sign[2:] < 0.0) & (close[2:] <= midpoint)
    )
    return (first_large & middle_small & last_large & opposite & passed).astype(
        np.float32
    )


def _soldiers(
    body_ratio: np.ndarray,
    upper_ratio: np.ndarray,
    lower_ratio: np.ndarray,
    sign: np.ndarray,
    close: np.ndarray,
) -> np.ndarray:
    """Three-soldier or three-crow bits for bars 2..n-1."""
    large = (
        (body_ratio[:-2] >= SOLDIER_BODY_MIN)
        & (body_ratio[1:-1] >= SOLDIER_BODY_MIN)
        & (body_ratio[2:] >= SOLDIER_BODY_MIN)
    )
    up = (
        (sign[:-2] > 0.0)
        & (sign[1:-1] > 0.0)
        & (sign[2:] > 0.0)
        & (close[1:-1] > close[:-2])
        & (close[2:] > close[1:-1])
        & (upper_ratio[:-2] < SOLDIER_WICK_MAX)
        & (upper_ratio[1:-1] < SOLDIER_WICK_MAX)
        & (upper_ratio[2:] < SOLDIER_WICK_MAX)
    )
    down = (
        (sign[:-2] < 0.0)
        & (sign[1:-1] < 0.0)
        & (sign[2:] < 0.0)
        & (close[1:-1] < close[:-2])
        & (close[2:] < close[1:-1])
        & (lower_ratio[:-2] < SOLDIER_WICK_MAX)
        & (lower_ratio[1:-1] < SOLDIER_WICK_MAX)
        & (lower_ratio[2:] < SOLDIER_WICK_MAX)
    )
    return (large & (up | down)).astype(np.float32)


def prior_context(close: np.ndarray) -> np.ndarray:
    """Context of bar ``t`` from closes ending at ``t - 1``.

    Args:
        close: Close prices.

    Returns:
        Object array of ``after_decline``, ``after_advance``, or ``after_flat``.
    """
    n = len(close)
    context = np.full(n, "after_flat", dtype=object)
    width = PRIOR_TREND_BARS
    if n <= width:
        return context
    kernel = np.ones(width, dtype=np.float64) / float(width)
    means = np.convolve(close.astype(np.float64), kernel, mode="valid")
    # means[k] ends at index k + width - 1. Bar t needs the mean ending at t - 1.
    for index in range(width, n):
        mean = float(means[index - width])
        previous = float(close[index - 1])
        if not np.isfinite(mean) or mean == 0.0:
            continue
        gap = (previous - mean) / mean
        if gap > TREND_FLAT_BAND:
            context[index] = "after_advance"
        elif gap < -TREND_FLAT_BAND:
            context[index] = "after_decline"
    return context


def build_rule_card(
    frame: pd.DataFrame,
    atr: np.ndarray,
    geometry: np.ndarray,
) -> pd.DataFrame:
    """Write conventional names beside geometric hits.

    Args:
        frame: OHLCV frame with a ``time`` column.
        atr: Causal ATR.
        geometry: Output of :func:`geometry_matrix`.

    Returns:
        One row per positive geometric class. These columns are not features.
    """
    open_ = frame["open"].to_numpy(dtype=np.float64)
    high = frame["high"].to_numpy(dtype=np.float64)
    low = frame["low"].to_numpy(dtype=np.float64)
    close = frame["close"].to_numpy(dtype=np.float64)
    context = prior_context(close)
    times = frame["time"].tolist()
    rows: list[dict[str, object]] = []
    for class_index, class_name in enumerate(GEOMETRIC_CLASSES):
        hits = np.flatnonzero(geometry[:, class_index] >= 0.5)
        for index in hits.tolist():
            name, bias, abstain = _interpret(
                class_name,
                int(index),
                open_,
                high,
                low,
                close,
                atr,
                str(context[index]),
            )
            rows.append(
                {
                    "bar_time": times[index],
                    "bar_index": int(index),
                    "geometric_class": class_name,
                    "conventional_name": name,
                    "directional_bias": bias,
                    "context": str(context[index]),
                    "abstain": bool(abstain),
                }
            )
    if not rows:
        return pd.DataFrame(columns=list(RULE_CARD_COLUMNS))
    card = pd.DataFrame(rows, columns=list(RULE_CARD_COLUMNS))
    return card


def _interpret(
    class_name: str,
    index: int,
    open_: np.ndarray,
    high: np.ndarray,
    low: np.ndarray,
    close: np.ndarray,
    atr: np.ndarray,
    context: str,
) -> tuple[str, str, bool]:
    """Return conventional name, bias, and abstain for one geometric hit."""
    sign = float(np.sign(close[index] - open_[index]))
    if class_name == "long_lower_wick":
        if context == "after_decline":
            return "hammer", "bullish", False
        if context == "after_advance":
            return "hanging_man", "bearish", False
        return "long_lower_wick", "context_dependent", True
    if class_name == "long_upper_wick":
        if context == "after_advance":
            return "shooting_star", "bearish", False
        if context == "after_decline":
            return "inverted_hammer", "context_dependent", True
        return "long_upper_wick", "context_dependent", True
    if class_name == "doji":
        return "doji", "neutral", False
    if class_name == "marubozu":
        return _signed_name("marubozu", sign)
    if class_name == "engulfing":
        return _signed_name("engulfing", sign)
    if class_name == "harami":
        return _signed_name("harami", sign)
    if class_name == "piercing_dark_cloud":
        if sign > 0.0:
            return "piercing", "bullish", False
        if sign < 0.0:
            return "dark_cloud", "bearish", False
        return "piercing_dark_cloud", "context_dependent", True
    if class_name == "tweezer":
        return _tweezer_name(index, high, low, atr)
    if class_name == "star":
        return _signed_name("star", sign)
    if class_name == "soldiers_crows":
        if sign > 0.0:
            return "three_white_soldiers", "bullish", False
        if sign < 0.0:
            return "three_black_crows", "bearish", False
        return "soldiers_crows", "context_dependent", True
    return class_name, "context_dependent", True


def _signed_name(stem: str, sign: float) -> tuple[str, str, bool]:
    """Attach bullish or bearish from the completing candle's body sign."""
    if sign > 0.0:
        return f"bullish_{stem}", "bullish", False
    if sign < 0.0:
        return f"bearish_{stem}", "bearish", False
    return stem, "context_dependent", True


def _tweezer_name(
    index: int,
    high: np.ndarray,
    low: np.ndarray,
    atr: np.ndarray,
) -> tuple[str, str, bool]:
    """Name a tweezer from which side matched. Both sides abstain."""
    if index < 1 or not np.isfinite(atr[index]) or atr[index] <= 0.0:
        return "tweezer", "context_dependent", True
    tol = TWEEZER_ATR_TOL * float(atr[index])
    top = abs(float(high[index] - high[index - 1])) <= tol
    bottom = abs(float(low[index] - low[index - 1])) <= tol
    if top and bottom:
        return "tweezer", "context_dependent", True
    if top:
        return "tweezer_top", "bearish", False
    if bottom:
        return "tweezer_bottom", "bullish", False
    return "tweezer", "context_dependent", True


def confirmed_swing_events(
    high: np.ndarray,
    low: np.ndarray,
    close: np.ndarray,
    atr: np.ndarray,
) -> list[tuple[int, int, str]]:
    """Causal ATR zigzag events.

    Args:
        high: High prices.
        low: Low prices.
        close: Close prices, used only to keep ATR positive.
        atr: Causal ATR.

    Returns:
        Tuples of ``(confirmation_index, extreme_index, kind)`` with kind in
        ``HH``, ``HL``, ``LH``, ``LL``. A swing is recorded only when price
        has reversed by ``ZZ_TAU`` ATR. Later bars are not read.
    """
    n = len(close)
    if n == 0:
        return []
    direction = 1
    extreme_index = 0
    extreme_price = float(high[0])
    last_high = float("nan")
    last_low = float("nan")
    events: list[tuple[int, int, str]] = []
    for index in range(n):
        atr_value = float(atr[index]) if np.isfinite(atr[index]) else float("nan")
        if not np.isfinite(atr_value) or atr_value <= 0.0:
            atr_value = max(abs(float(close[index])) * 0.01, 1e-9)
        threshold = ZZ_TAU * atr_value
        band = ZZ_EPS_ATR * atr_value
        if direction == 1:
            if float(high[index]) >= extreme_price:
                extreme_price = float(high[index])
                extreme_index = index
            elif (extreme_price - float(low[index])) >= threshold:
                kind = _swing_kind(extreme_price, last_high, band, "HH", "LH")
                if kind is not None:
                    events.append((index, extreme_index, kind))
                last_high = extreme_price
                direction = -1
                extreme_price = float(low[index])
                extreme_index = index
        else:
            if float(low[index]) <= extreme_price:
                extreme_price = float(low[index])
                extreme_index = index
            elif (float(high[index]) - extreme_price) >= threshold:
                kind = _swing_kind(extreme_price, last_low, band, "HL", "LL")
                if kind is not None:
                    events.append((index, extreme_index, kind))
                last_low = extreme_price
                direction = 1
                extreme_price = float(high[index])
                extreme_index = index
    return events


def _swing_kind(
    price: float,
    previous: float,
    band: float,
    higher_name: str,
    lower_name: str,
) -> str | None:
    """Compare a confirmed extreme with the previous extreme of the same side."""
    if not np.isfinite(previous):
        return None
    if price > previous + band:
        return higher_name
    if price < previous - band:
        return lower_name
    return None


def market_labels(
    high: np.ndarray,
    low: np.ndarray,
    close: np.ndarray,
    atr: np.ndarray,
    *,
    window: int = WINDOW_BARS,
) -> tuple[np.ndarray, np.ndarray]:
    """Window-level market state and structure tag at each bar.

    Args:
        high: High prices.
        low: Low prices.
        close: Close prices.
        atr: Causal ATR.
        window: Bars visible to the label, including the current bar.

    Returns:
        Integer state ids and tag ids. A swing counts only when its extreme
        and its confirmation bar both sit inside the window ending at ``t``.
    """
    n = len(close)
    states = np.full(n, _STATE_INDEX["indeterminate"], dtype=np.int64)
    tags = np.full(n, _TAG_INDEX["none"], dtype=np.int64)
    events = confirmed_swing_events(high, low, close, atr)
    left = 0
    right = 0
    for end in range(n):
        start = end - window + 1
        while right < len(events) and events[right][0] <= end:
            right += 1
        while left < right and events[left][0] < start:
            left += 1
        visible = [
            kind
            for confirm, extreme, kind in events[left:right]
            if extreme >= start and confirm <= end
        ]
        state_name, tag_name = _state_from_events(visible)
        states[end] = _STATE_INDEX[state_name]
        tags[end] = _TAG_INDEX[tag_name]
    return states, tags


def _state_from_events(kinds: Iterable[str]) -> tuple[str, str]:
    """Map visible swing events to a market state and a structure tag."""
    sequence = list(kinds)
    if len(sequence) < 2:
        return "indeterminate", "none"
    bull = sum(kind in ("HH", "HL") for kind in sequence)
    bear = sum(kind in ("LH", "LL") for kind in sequence)
    last = sequence[-1]
    if last in ("LH", "LL") and bull >= 2 and bull > bear:
        return "transition", "hh_hl"
    if last in ("HH", "HL") and bear >= 2 and bear > bull:
        return "transition", "lh_ll"
    if bull >= 2 and bull > bear:
        return "uptrend", "hh_hl"
    if bear >= 2 and bear > bull:
        return "downtrend", "lh_ll"
    return "range_bound", "mixed"


## Purged splits

In [ ]:
"""Chronological splits with a purge of one full recognition window."""

from __future__ import annotations

import numpy as np



def eligible_ends(n_bars: int, first_finite: int, window: int = WINDOW_BARS) -> np.ndarray:
    """Bar indices whose window lies entirely on finite feature rows.

    Args:
        n_bars: Length of the 15-minute series.
        first_finite: First bar with finite engineered features.
        window: Window length.

    Returns:
        Increasing end indices. Empty when the series is too short.
    """
    first_end = first_finite + window - 1
    if first_end >= n_bars:
        return np.empty(0, dtype=np.int64)
    return np.arange(first_end, n_bars, dtype=np.int64)


def purged_split(
    end_indices: np.ndarray,
    *,
    purge: int = PURGE_BARS,
    train_fraction: float = TRAIN_FRACTION,
    val_fraction: float = VAL_FRACTION,
) -> dict[str, np.ndarray]:
    """Split end indices in time, leaving ``purge`` bars between partitions.

    Args:
        end_indices: Bar indices of window ends, sorted ascending. The values
            are bar positions, so a gap of ``purge`` bars means the next
            window starts after the previous window ends.
        purge: Minimum bar gap between the last end of one split and the
            first end of the next.
        train_fraction: Share of ends assigned to train before the purge.
        val_fraction: Share of ends assigned to validation before the purge.

    Returns:
        Train, validation, and test end indices.

    Raises:
        ValueError: If a partition is empty or the gap is shorter than
            ``purge`` bars.
    """
    ends = np.asarray(end_indices, dtype=np.int64)
    if ends.size == 0 or np.any(np.diff(ends) < 0):
        raise ValueError("end indices must be non-empty and sorted")
    train_count = int(ends.size * train_fraction)
    val_count = int(ends.size * val_fraction)
    if train_count < 1 or val_count < 1:
        raise ValueError("series is too short to allocate train and validation")
    train = ends[:train_count]
    val_start = int(np.searchsorted(ends, int(train[-1]) + purge, side="left"))
    val = ends[val_start : val_start + val_count]
    if val.size == 0:
        raise ValueError("purge removed the validation partition")
    test_start = int(np.searchsorted(ends, int(val[-1]) + purge, side="left"))
    test = ends[test_start:]
    if test.size == 0:
        raise ValueError("purge removed the test partition")
    _require_gap(train, val, purge)
    _require_gap(val, test, purge)
    return {"train": train, "val": val, "test": test}


def _require_gap(left: np.ndarray, right: np.ndarray, purge: int) -> None:
    """Raise when two partitions share a window or sit closer than ``purge``."""
    gap = int(right[0]) - int(left[-1])
    if gap < purge:
        raise ValueError(f"split gap {gap} is shorter than the purge of {purge}")


def development_folds(
    train_ends: np.ndarray,
    val_ends: np.ndarray,
    *,
    folds: int = 3,
    purge: int = PURGE_BARS,
) -> list[dict[str, np.ndarray]]:
    """Build purged chronological folds inside the development span.

    Args:
        train_ends: Outer training window ends. The sealed test is absent.
        val_ends: Outer validation window ends. The sealed test is absent.
        folds: Number of validation blocks.
        purge: Minimum bar gap between a fold's train end and its validation.

    Returns:
        One mapping of ``train`` and ``val`` end indices per fold.

    Raises:
        ValueError: If a fold cannot keep the purge.
    """
    if folds < 1:
        raise ValueError("folds must be positive")
    dev = np.sort(np.unique(np.concatenate([
        np.asarray(train_ends, dtype=np.int64),
        np.asarray(val_ends, dtype=np.int64),
    ])))
    blocks = folds + 1
    if dev.size < blocks * 2:
        raise ValueError("development span is too short for purged folds")
    chunk = dev.size // blocks
    if chunk < 1:
        raise ValueError("development span is too short for purged folds")
    built: list[dict[str, np.ndarray]] = []
    for index in range(folds):
        val_pos = chunk * (index + 1)
        val_stop = chunk * (index + 2) if index < folds - 1 else dev.size
        val = dev[val_pos:val_stop]
        if val.size == 0:
            raise ValueError("a development fold has no validation windows")
        train = dev[dev <= int(val[0]) - purge]
        if train.size == 0:
            raise ValueError("a development fold has no training windows")
        _require_gap(train, val, purge)
        built.append({"train": train, "val": val})
    return built


def strided_ends(end_indices: np.ndarray, stride: int = TRAIN_STRIDE) -> np.ndarray:
    """Take every ``stride`` window end, preserving chronological order."""
    ends = np.asarray(end_indices, dtype=np.int64)
    if stride < 1:
        raise ValueError("stride must be positive")
    return ends[::stride]


def nonoverlapping_ends(
    end_indices: np.ndarray,
    stride: int = EVAL_STRIDE,
) -> np.ndarray:
    """Keep window ends that are at least ``stride`` bars apart.

    Args:
        end_indices: Sorted bar indices.
        stride: Minimum gap. The recognition window uses 48.

    Returns:
        Subset of ``end_indices``. The first end is kept.
    """
    ends = np.asarray(end_indices, dtype=np.int64)
    if ends.size == 0:
        return ends
    kept = [int(ends[0])]
    for end in ends[1:]:
        if int(end) - kept[-1] >= stride:
            kept.append(int(end))
    return np.asarray(kept, dtype=np.int64)


## Leakage checks

In [ ]:
"""Guards that stop pattern flags and future bars from entering training."""

from __future__ import annotations

from typing import Iterable, Sequence

import numpy as np
import pandas as pd



def assert_feature_names(names: Sequence[str]) -> None:
    """Reject channels outside the allow-list or on the denylist.

    Args:
        names: Channel names about to be placed in the training tensor.

    Raises:
        ValueError: If a name is unknown, forbidden, or repeated.
    """
    if len(set(names)) != len(names):
        raise ValueError(f"duplicate feature channels: {list(names)}")
    blocked = forbidden_names(names)
    if blocked:
        raise ValueError(f"channels leak labels or forecast targets: {blocked}")
    unknown = [name for name in names if name not in ALLOWED_CHANNELS]
    if unknown:
        raise ValueError(f"channels are outside the recognition allow-list: {unknown}")


def forbidden_names(names: Iterable[str]) -> list[str]:
    """Return names that match a forbidden prefix or an exact denylisted column."""
    blocked: list[str] = []
    for name in names:
        if name in FORBIDDEN_EXACT or name.startswith(FORBIDDEN_PREFIXES):
            blocked.append(name)
    return blocked


def assert_causal_at(frame: pd.DataFrame, index: int) -> None:
    """Require features and labels at ``index`` to ignore later bars.

    Args:
        frame: 15-minute OHLCV with a ``time`` column.
        index: Bar whose outputs must stay fixed when later bars change.

    Raises:
        ValueError: If a feature, geometric label, or market state changes.
        IndexError: If ``index`` is outside the frame.
    """
    if index < 0 or index >= len(frame):
        raise IndexError(f"index {index} is outside the frame of length {len(frame)}")
    mutated = frame.copy()
    later = mutated.index[index + 1 :]
    if len(later) == 0:
        raise ValueError("perturbation requires bars after the inspected index")
    for column in ("open", "high", "low", "close", "volume"):
        mutated.loc[later, column] = mutated.loc[later, column].astype(float) * 1.7

    original_features = add_engineered_features(frame)
    mutated_features = add_engineered_features(mutated)
    computed = [column for column in original_features.columns if column != "time"]
    left = original_features.iloc[: index + 1].loc[:, computed].to_numpy(dtype=np.float64)
    right = mutated_features.iloc[: index + 1].loc[:, computed].to_numpy(dtype=np.float64)
    if not np.allclose(left, right, equal_nan=True):
        raise ValueError("engineered features changed when future bars were scaled")

    original_geometry = _geometry_from(original_features)
    mutated_geometry = _geometry_from(mutated_features)
    if not np.array_equal(original_geometry[: index + 1], mutated_geometry[: index + 1]):
        raise ValueError("geometric labels changed when future bars were scaled")

    original_state, original_tag = _market_from(original_features)
    mutated_state, mutated_tag = _market_from(mutated_features)
    if not np.array_equal(original_state[: index + 1], mutated_state[: index + 1]):
        raise ValueError("market state changed when future bars were scaled")
    if not np.array_equal(original_tag[: index + 1], mutated_tag[: index + 1]):
        raise ValueError("structure tag changed when future bars were scaled")


def _geometry_from(features: pd.DataFrame) -> np.ndarray:
    """Geometric labels from a feature frame's prices and ATR."""
    return geometry_matrix(
        features["open"].to_numpy(dtype=np.float64),
        features["high"].to_numpy(dtype=np.float64),
        features["low"].to_numpy(dtype=np.float64),
        features["close"].to_numpy(dtype=np.float64),
        features["atr"].to_numpy(dtype=np.float64),
    )


def _market_from(features: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    """Market state and tag from a feature frame's prices and ATR."""
    return market_labels(
        features["high"].to_numpy(dtype=np.float64),
        features["low"].to_numpy(dtype=np.float64),
        features["close"].to_numpy(dtype=np.float64),
        features["atr"].to_numpy(dtype=np.float64),
    )


## Windows

In [ ]:
"""Window tensors for the recognition track.

The rule card is built so it can be audited, then kept off the feature tensor.
"""

from __future__ import annotations

from typing import Mapping

import numpy as np
import pandas as pd



class ChannelScaler:
    """Per-channel mean and standard deviation fit on training windows only."""

    def __init__(self) -> None:
        self.mean = np.zeros(1, dtype=np.float32)
        self.std = np.ones(1, dtype=np.float32)

    def fit(self, windows: np.ndarray) -> "ChannelScaler":
        """Fit on an array of shape ``(n, time, channels)``.

        Args:
            windows: Training windows.

        Returns:
            This scaler.
        """
        if windows.ndim != 3:
            raise ValueError("windows must have shape (n, time, channels)")
        mean = windows.mean(axis=(0, 1)).astype(np.float32)
        std = windows.std(axis=(0, 1)).astype(np.float32)
        std = np.where(std < 1e-6, np.float32(1.0), std)
        self.mean = mean
        self.std = std.astype(np.float32)
        return self

    def transform(self, windows: np.ndarray) -> np.ndarray:
        """Standardize with the training mean and standard deviation."""
        return ((windows - self.mean) / self.std).astype(np.float32)


def prepare_series(frame: pd.DataFrame) -> dict[str, object]:
    """Compute features, labels, the rule card, and purged splits.

    Args:
        frame: Completed 15-minute OHLCV.

    Returns:
        Mapping with feature frame, geometry, market labels, rule card, and
        split end indices. The rule card is not a feature matrix.

    Raises:
        ValueError: If the rule card shares a column with the model channels.
    """
    features = add_engineered_features(frame)
    open_ = features["open"].to_numpy(dtype=np.float64)
    high = features["high"].to_numpy(dtype=np.float64)
    low = features["low"].to_numpy(dtype=np.float64)
    close = features["close"].to_numpy(dtype=np.float64)
    atr = features["atr"].to_numpy(dtype=np.float64)
    geometry = geometry_matrix(open_, high, low, close, atr)
    states, tags = market_labels(high, low, close, atr)
    rule_card = build_rule_card(features, atr, geometry)
    leaked = set(rule_card.columns) & (set(ENGINEERED_CHANNELS) | set(RAW_CHANNELS))
    if leaked:
        raise ValueError(f"rule card columns entered the feature set: {sorted(leaked)}")
    assert_feature_names(RAW_CHANNELS)
    assert_feature_names(ENGINEERED_CHANNELS)
    ends = eligible_ends(len(features), first_finite_bar(features))
    splits = purged_split(ends)
    return {
        "features": features,
        "geometry": geometry,
        "states": states,
        "tags": tags,
        "rule_card": rule_card,
        "splits": splits,
    }


def stack_windows(
    prepared: Mapping[str, object],
    end_indices: np.ndarray,
    mode: str,
) -> dict[str, np.ndarray]:
    """Stack a causal window for each end index.

    Args:
        prepared: Output of :func:`prepare_series`.
        end_indices: Window end bar indices.
        mode: ``raw`` or ``engineered``.

    Returns:
        ``features`` of shape ``(n, 48, channels)``, ``geometry`` of shape
        ``(n, 48, classes)``, and integer ``state`` and ``tag`` vectors.

    Raises:
        ValueError: If ``mode`` is unknown or a window is non-finite.
    """
    ends = np.asarray(end_indices, dtype=np.int64)
    features = prepared["features"]
    if not isinstance(features, pd.DataFrame):
        raise TypeError("prepared series is missing the feature frame")
    if mode == "raw":
        windows = _raw_windows(features, ends)
        channels = RAW_CHANNELS
    elif mode == "engineered":
        windows = _engineered_windows(features, ends)
        channels = ENGINEERED_CHANNELS
    else:
        raise ValueError(f"unknown feature mode: {mode}")
    assert_feature_names(channels)
    if not np.isfinite(windows).all():
        raise ValueError(f"{mode} windows contain non-finite values")
    geometry = np.asarray(prepared["geometry"])
    grid = ends[:, None] - (WINDOW_BARS - 1) + np.arange(WINDOW_BARS)[None, :]
    return {
        "features": windows,
        "geometry": geometry[grid].astype(np.float32),
        "state": np.asarray(prepared["states"])[ends].astype(np.int64),
        "tag": np.asarray(prepared["tags"])[ends].astype(np.int64),
        "channels": np.asarray(channels),
    }


def _engineered_windows(features: pd.DataFrame, ends: np.ndarray) -> np.ndarray:
    """Slice engineered channels into ``(n, time, channels)`` windows."""
    matrix = features.loc[:, list(ENGINEERED_CHANNELS)].to_numpy(dtype=np.float32)
    grid = ends[:, None] - (WINDOW_BARS - 1) + np.arange(WINDOW_BARS)[None, :]
    return matrix[grid]


def _raw_windows(features: pd.DataFrame, ends: np.ndarray) -> np.ndarray:
    """Scale raw OHLCV by the ATR and volume baseline at the window end."""
    open_ = features["open"].to_numpy(dtype=np.float64)
    high = features["high"].to_numpy(dtype=np.float64)
    low = features["low"].to_numpy(dtype=np.float64)
    close = features["close"].to_numpy(dtype=np.float64)
    volume = features["volume"].to_numpy(dtype=np.float64)
    atr = features["atr"].to_numpy(dtype=np.float64)
    baseline = features["volume_baseline"].to_numpy(dtype=np.float64)
    starts = ends - (WINDOW_BARS - 1)
    grid = starts[:, None] + np.arange(WINDOW_BARS)[None, :]
    atr_ref = np.array([safe_scale(float(atr[end])) for end in ends], dtype=np.float64)
    vol_ref = np.array(
        [safe_scale(float(baseline[end])) for end in ends],
        dtype=np.float64,
    )
    reference = close[starts]
    prices = (
        np.stack([open_[grid], high[grid], low[grid], close[grid]], axis=-1)
        - reference[:, None, None]
    ) / atr_ref[:, None, None]
    volume_ratio = volume[grid] / vol_ref[:, None]
    return np.concatenate(
        [prices.astype(np.float32), volume_ratio.astype(np.float32)[..., None]],
        axis=-1,
    )


def rule_card_is_separate(prepared: Mapping[str, object]) -> bool:
    """Return True when the rule card shares no column with model channels."""
    card = prepared["rule_card"]
    if not isinstance(card, pd.DataFrame):
        return False
    if list(card.columns) != list(RULE_CARD_COLUMNS):
        return False
    return not (set(card.columns) & (set(RAW_CHANNELS) | set(ENGINEERED_CHANNELS)))


## Causal TCN

In [ ]:
"""Phase-1 trainers: a causal TCN and an XGBoost presence baseline.

Head A is per-bar geometric multi-label. Head E is the window's market state
and structure tag. Direction, chart patterns, formation status, breakouts,
and zone geometry are not trained.
"""

from __future__ import annotations

from typing import Any

import numpy as np
import structlog
import torch
from torch import nn
from xgboost import XGBClassifier

logger = structlog.get_logger(__name__)



class CausalConv1d(nn.Module):
    """Convolution that pads on the left, so each step sees only the past."""

    def __init__(self, in_channels: int, out_channels: int, kernel: int, dilation: int) -> None:
        super().__init__()
        self.pad = (kernel - 1) * dilation
        self.conv = nn.Conv1d(
            in_channels,
            out_channels,
            kernel,
            dilation=dilation,
        )

    def forward(self, values: torch.Tensor) -> torch.Tensor:
        """Apply the convolution to ``(n, channels, time)``."""
        padded = nn.functional.pad(values, (self.pad, 0))
        return self.conv(padded)


class RecognitionTCN(nn.Module):
    """Shared causal encoder with geometric and market-state heads."""

    def __init__(
        self,
        in_channels: int,
        n_geometry: int,
        n_states: int,
        n_tags: int,
        hidden: int = TCN_HIDDEN,
        dilations: tuple[int, ...] = TCN_DILATIONS,
    ) -> None:
        super().__init__()
        blocks: list[nn.Module] = []
        channels = in_channels
        for dilation in dilations:
            blocks.append(CausalConv1d(channels, hidden, TCN_KERNEL, int(dilation)))
            blocks.append(nn.ReLU())
            channels = hidden
        self.encoder = nn.Sequential(*blocks)
        self.head_geometry = nn.Conv1d(hidden, n_geometry, kernel_size=1)
        self.head_state = nn.Linear(hidden, n_states)
        self.head_tag = nn.Linear(hidden, n_tags)

    def forward(self, values: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Return geometry logits, state logits, and tag logits.

        Args:
            values: ``(n, time, channels)``.

        Returns:
            Geometry logits ``(n, time, classes)`` and two window logits
            taken from the last timestep.
        """
        hidden = self.encoder(values.transpose(1, 2))
        geometry = self.head_geometry(hidden).transpose(1, 2)
        pooled = hidden[:, :, -1]
        return geometry, self.head_state(pooled), self.head_tag(pooled)


def train_tcn(
    train: dict[str, np.ndarray],
    val: dict[str, np.ndarray],
    test: dict[str, np.ndarray] | None = None,
    *,
    epochs: int = TCN_EPOCHS,
    batch_size: int = TCN_BATCH,
    seed: int = SEED,
    hidden: int = TCN_HIDDEN,
    dilations: tuple[int, ...] = TCN_DILATIONS,
    learning_rate: float = TCN_LEARNING_RATE,
    pos_weight_clip: float = POS_WEIGHT_CLIP,
    patience: int | None = None,
    include_test_calibration: bool = False,
) -> dict[str, Any]:
    """Fit Head A and Head E. The epoch is chosen on validation loss.

    Args:
        train: Scaled windows and labels.
        val: Validation windows used only to choose the epoch.
        test: Held-out windows. Scored once after the epoch is chosen.
            Omitted during the development search.
        epochs: Maximum passes over the training windows.
        batch_size: Mini-batch size.
        seed: Torch seed.
        hidden: Encoder width.
        dilations: Causal dilation schedule.
        learning_rate: Step size for AdamW or the clipped SGD fallback.
        pos_weight_clip: Upper bound on the Head A positive-class weight.
        patience: Stop after this many epochs without a lower validation
            loss. ``None`` runs every epoch and still keeps the best one.
        include_test_calibration: When scoring ``test``, add Head A
            reliability bins. This is recognition calibration.

    Returns:
        Metrics for validation and, when provided, test, plus the CPU
        ``state_dict`` of the chosen epoch. ``best_epoch`` is the
        zero-based index of the lowest validation loss.
    """
    torch.manual_seed(seed)
    model = RecognitionTCN(
        in_channels=int(train["features"].shape[-1]),
        n_geometry=len(GEOMETRIC_CLASSES),
        n_states=len(MARKET_STATES),
        n_tags=len(STRUCTURE_TAGS),
        hidden=hidden,
        dilations=dilations,
    )
    optimizer = _adamw(model, learning_rate)
    pos_weight = _as_tensor(_positive_weights(train["geometry"], pos_weight_clip))
    state_weight = _as_tensor(_class_weights(train["state"], len(MARKET_STATES)))
    tag_weight = _as_tensor(_class_weights(train["tag"], len(STRUCTURE_TAGS)))
    best_loss = float("inf")
    best_epoch = 0
    best_state: dict[str, torch.Tensor] | None = None
    train_history: list[float] = []
    history: list[float] = []
    stale = 0
    for epoch in range(epochs):
        train_loss = _run_epoch(
            model,
            train,
            pos_weight,
            state_weight,
            tag_weight,
            batch_size,
            seed + epoch,
            learning_rate,
            optimizer,
        )
        val_loss = _loss(
            model,
            val,
            pos_weight,
            state_weight,
            tag_weight,
            batch_size,
        )
        train_history.append(train_loss)
        history.append(val_loss)
        logger.info(
            "recognition_epoch",
            epoch=epoch,
            train_loss=train_loss,
            val_loss=val_loss,
        )
        if val_loss < best_loss:
            best_loss = val_loss
            best_epoch = epoch
            stale = 0
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
        else:
            stale += 1
            if patience is not None and stale >= patience:
                break
    if best_state is None:
        raise RuntimeError("TCN training produced no epoch")
    model.load_state_dict(best_state)
    scored_test = None
    if test is not None:
        scored_test = _score(
            model,
            test,
            batch_size,
            include_calibration=include_test_calibration,
        )
    return {
        "val_loss": best_loss,
        "best_epoch": best_epoch,
        "train_loss_by_epoch": train_history,
        "val_loss_by_epoch": history,
        "val": _score(model, val, batch_size),
        "test": scored_test,
        "state_dict": best_state,
        "optimizer": "adamw" if optimizer is not None else "sgd",
    }


def train_presence_baseline(
    train_features: np.ndarray,
    train_geometry: np.ndarray,
    test_features: np.ndarray,
    test_geometry: np.ndarray,
    *,
    seed: int = SEED,
) -> dict[str, Any]:
    """Fit one tree per geometric family on window-level presence.

    Args:
        train_features: ``(n, time, channels)`` training windows.
        train_geometry: Per-bar multi-label targets.
        test_features: Held-out windows.
        test_geometry: Held-out per-bar targets.
        seed: Tree seed.

    Returns:
        Per-class presence metrics. Classes without both outcomes in train
        are recorded as skipped. Spans are not predicted.
    """
    x_train = train_features.reshape(len(train_features), -1)
    x_test = test_features.reshape(len(test_features), -1)
    y_train = (train_geometry.max(axis=1) >= 0.5).astype(np.int64)
    y_test = (test_geometry.max(axis=1) >= 0.5).astype(np.int64)
    per_class: dict[str, Any] = {}
    f1_scores: list[float] = []
    for class_index, name in enumerate(GEOMETRIC_CLASSES):
        labels = y_train[:, class_index]
        positives = int(labels.sum())
        negatives = int(len(labels) - positives)
        if positives == 0 or negatives == 0:
            per_class[name] = {"skipped": True, "support": int(y_test[:, class_index].sum())}
            continue
        model = XGBClassifier(
            n_estimators=XGB_ESTIMATORS,
            max_depth=XGB_MAX_DEPTH,
            learning_rate=0.1,
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=float(negatives) / float(positives),
            tree_method="hist",
            n_jobs=1,
            random_state=seed,
        )
        model.fit(x_train, labels)
        scores = model.predict_proba(x_test)[:, 1]
        metrics = binary_scores(y_test[:, class_index], scores)
        metrics["skipped"] = False
        per_class[name] = metrics
        if metrics["support"] > 0:
            f1_scores.append(metrics["f1"])
    macro = float(np.mean(f1_scores)) if f1_scores else 0.0
    return {"per_class": per_class, "macro_f1": macro}


def binary_scores(
    y_true: np.ndarray,
    y_score: np.ndarray,
    threshold: float = 0.5,
) -> dict[str, float]:
    """Precision, recall, and F1 for one binary label."""
    pred = y_score >= threshold
    true = y_true.astype(bool)
    true_positive = int(np.logical_and(pred, true).sum())
    false_positive = int(np.logical_and(pred, ~true).sum())
    false_negative = int(np.logical_and(~pred, true).sum())
    precision = _ratio(true_positive, true_positive + false_positive)
    recall = _ratio(true_positive, true_positive + false_negative)
    return {
        "precision": precision,
        "recall": recall,
        "f1": _f1(precision, recall),
        "support": float(true.sum()),
    }


def multiclass_scores(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    names: tuple[str, ...],
) -> dict[str, Any]:
    """Macro-F1 over classes that appear in ``y_true``, plus per-class recall."""
    per_class: dict[str, dict[str, float]] = {}
    f1_scores: list[float] = []
    for class_index, name in enumerate(names):
        true = y_true == class_index
        pred = y_pred == class_index
        true_positive = int(np.logical_and(pred, true).sum())
        false_positive = int(np.logical_and(pred, ~true).sum())
        false_negative = int(np.logical_and(~pred, true).sum())
        precision = _ratio(true_positive, true_positive + false_positive)
        recall = _ratio(true_positive, true_positive + false_negative)
        score = _f1(precision, recall)
        per_class[name] = {
            "precision": precision,
            "recall": recall,
            "f1": score,
            "support": float(true.sum()),
        }
        if int(true.sum()) > 0:
            f1_scores.append(score)
    macro = float(np.mean(f1_scores)) if f1_scores else 0.0
    return {"macro_f1": macro, "per_class": per_class}


def _positive_weights(geometry: np.ndarray, clip: float = POS_WEIGHT_CLIP) -> np.ndarray:
    """Positive-class weights for binary cross-entropy, clipped."""
    rates = geometry.reshape(-1, geometry.shape[-1]).mean(axis=0)
    rates = np.clip(rates, 1e-4, 1.0 - 1e-4)
    weights = (1.0 - rates) / rates
    return np.clip(weights, 1.0, clip).astype(np.float32)


def _class_weights(labels: np.ndarray, n_classes: int) -> np.ndarray:
    """Inverse-frequency weights. Missing classes receive weight zero."""
    counts = np.bincount(labels.astype(np.int64), minlength=n_classes).astype(np.float64)
    weights = np.zeros(n_classes, dtype=np.float32)
    present = counts > 0.0
    if present.any():
        weights[present] = counts.sum() / (float(present.sum()) * counts[present])
    return weights


def _run_epoch(
    model: RecognitionTCN,
    batch: dict[str, np.ndarray],
    pos_weight: torch.Tensor,
    state_weight: torch.Tensor,
    tag_weight: torch.Tensor,
    batch_size: int,
    seed: int,
    learning_rate: float,
    optimizer: torch.optim.Optimizer | None,
) -> float:
    """One shuffled pass over the training split.

    Returns:
        Mean training loss for the pass, weighted by batch size.
    """
    model.train()
    order = np.arange(len(batch["features"]))
    rng = np.random.default_rng(seed)
    rng.shuffle(order)
    total = 0.0
    count = 0
    for start in range(0, len(order), batch_size):
        chosen = order[start : start + batch_size]
        if optimizer is not None:
            optimizer.zero_grad(set_to_none=True)
        else:
            model.zero_grad(set_to_none=True)
        loss = _batch_loss(
            model,
            batch,
            chosen,
            pos_weight,
            state_weight,
            tag_weight,
        )
        loss.backward()
        if optimizer is None:
            _sgd_step(model, learning_rate)
        else:
            optimizer.step()
        total += float(loss.detach().cpu()) * len(chosen)
        count += len(chosen)
    if count == 0:
        return float("inf")
    return total / count


def _loss(
    model: RecognitionTCN,
    batch: dict[str, np.ndarray],
    pos_weight: torch.Tensor,
    state_weight: torch.Tensor,
    tag_weight: torch.Tensor,
    batch_size: int,
) -> float:
    """Mean loss over ``batch`` without updating weights."""
    model.eval()
    total = 0.0
    count = 0
    with torch.no_grad():
        for start in range(0, len(batch["features"]), batch_size):
            chosen = np.arange(start, min(start + batch_size, len(batch["features"])))
            loss = _batch_loss(
                model,
                batch,
                chosen,
                pos_weight,
                state_weight,
                tag_weight,
            )
            total += float(loss.detach().cpu()) * len(chosen)
            count += len(chosen)
    if count == 0:
        return float("inf")
    return total / count


def _batch_loss(
    model: RecognitionTCN,
    batch: dict[str, np.ndarray],
    chosen: np.ndarray,
    pos_weight: torch.Tensor,
    state_weight: torch.Tensor,
    tag_weight: torch.Tensor,
) -> torch.Tensor:
    """Sum of Head A and Head E losses. Task weights stay at one."""
    features = _as_tensor(batch["features"][chosen].astype(np.float32))
    geometry = _as_tensor(batch["geometry"][chosen].astype(np.float32))
    state = _as_tensor(batch["state"][chosen].astype(np.int64))
    tag = _as_tensor(batch["tag"][chosen].astype(np.int64))
    geometry_logits, state_logits, tag_logits = model(features)
    geometry_loss = nn.functional.binary_cross_entropy_with_logits(
        geometry_logits,
        geometry,
        pos_weight=pos_weight,
    )
    state_loss = nn.functional.cross_entropy(state_logits, state, weight=state_weight)
    tag_loss = nn.functional.cross_entropy(tag_logits, tag, weight=tag_weight)
    return geometry_loss + state_loss + tag_loss


def _score(
    model: RecognitionTCN,
    batch: dict[str, np.ndarray],
    batch_size: int = TCN_BATCH,
    *,
    include_calibration: bool = False,
) -> dict[str, Any]:
    """Recognition metrics on one split."""
    model.eval()
    geometry_scores = []
    state_pred = []
    tag_pred = []
    with torch.no_grad():
        for start in range(0, len(batch["features"]), batch_size):
            chosen = np.arange(start, min(start + batch_size, len(batch["features"])))
            features = _as_tensor(batch["features"][chosen].astype(np.float32))
            geometry_logits, state_logits, tag_logits = model(features)
            geometry_scores.append(_to_numpy(torch.sigmoid(geometry_logits)))
            state_pred.append(_to_numpy(torch.argmax(state_logits, dim=1)).astype(np.int64))
            tag_pred.append(_to_numpy(torch.argmax(tag_logits, dim=1)).astype(np.int64))
    geometry = np.concatenate(geometry_scores, axis=0)
    true_geometry = batch["geometry"].reshape(-1, geometry.shape[-1])
    pred_geometry = geometry.reshape(-1, geometry.shape[-1])
    per_class = {
        name: binary_scores(true_geometry[:, index], pred_geometry[:, index])
        for index, name in enumerate(GEOMETRIC_CLASSES)
    }
    present = [scores["f1"] for scores in per_class.values() if scores["support"] > 0.0]
    macro = float(np.mean(present)) if present else 0.0
    scored: dict[str, Any] = {
        "geometry_macro_f1": macro,
        "geometry_per_class": per_class,
        "market_state": multiclass_scores(
            batch["state"],
            np.concatenate(state_pred),
            MARKET_STATES,
        ),
        "structure_tag": multiclass_scores(
            batch["tag"],
            np.concatenate(tag_pred),
            STRUCTURE_TAGS,
        ),
    }
    if include_calibration:
        scored["geometry_calibration"] = calibration_table(true_geometry, pred_geometry)
    return scored


def calibration_table(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    n_bins: int = 10,
) -> dict[str, Any]:
    """Expected calibration error for recognition probabilities.

    Args:
        y_true: Binary labels, any shape.
        y_prob: Probabilities aligned with ``y_true``.
        n_bins: Equal-width bins on ``[0, 1]``.

    Returns:
        The error and a reliability table. This is not a trading metric.
    """
    true = np.asarray(y_true, dtype=np.float64).reshape(-1)
    prob = np.asarray(y_prob, dtype=np.float64).reshape(-1)
    if true.shape != prob.shape:
        raise ValueError("calibration labels and probabilities differ in shape")
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    rows: list[dict[str, float]] = []
    error = 0.0
    total = max(len(true), 1)
    for index in range(n_bins):
        lower = float(edges[index])
        upper = float(edges[index + 1])
        if index == n_bins - 1:
            mask = (prob >= lower) & (prob <= upper)
        else:
            mask = (prob >= lower) & (prob < upper)
        count = int(mask.sum())
        if count == 0:
            continue
        confidence = float(prob[mask].mean())
        frequency = float(true[mask].mean())
        error += (count / total) * abs(frequency - confidence)
        rows.append(
            {
                "lower": lower,
                "upper": upper,
                "count": float(count),
                "confidence": confidence,
                "frequency": frequency,
            }
        )
    return {"expected_calibration_error": float(error), "bins": rows}


def _adamw(model: nn.Module, learning_rate: float) -> torch.optim.Optimizer | None:
    """Build AdamW, or return None when this torch install cannot."""
    try:
        return torch.optim.AdamW(model.parameters(), lr=learning_rate)
    except Exception:
        logger.info("recognition_optimizer_fallback", optimizer="sgd")
        return None


def _as_tensor(values: np.ndarray) -> torch.Tensor:
    """Copy a NumPy array into a tensor without the NumPy bridge.

    The installed torch build was compiled against NumPy 1.x and cannot infer
    dtypes from NumPy 2 arrays. The byte copy keeps training on this machine.
    """
    array = np.ascontiguousarray(values)
    if array.dtype == np.float64:
        array = np.ascontiguousarray(array.astype(np.float32))
    if array.dtype == np.float32:
        dtype = torch.float32
    elif array.dtype == np.int64:
        dtype = torch.int64
    else:
        raise TypeError(f"unsupported array dtype: {array.dtype}")
    raw = bytearray(array.tobytes())
    return torch.frombuffer(raw, dtype=dtype).reshape(array.shape).clone()


def _to_numpy(values: torch.Tensor) -> np.ndarray:
    """Copy a CPU tensor to NumPy via a Python list."""
    detached = values.detach().cpu()
    return np.asarray(detached.reshape(-1).tolist(), dtype=np.float64).reshape(detached.shape)


def _sgd_step(model: nn.Module, learning_rate: float) -> None:
    """Gradient step with norm clipping. Avoids ``torch.optim`` on this install."""
    squared = None
    for parameter in model.parameters():
        if parameter.grad is None:
            continue
        term = parameter.grad.detach().pow(2).sum()
        squared = term if squared is None else squared + term
    if squared is None:
        return
    scale = max(1.0, float(squared.sqrt()))
    with torch.no_grad():
        for parameter in model.parameters():
            if parameter.grad is None:
                continue
            parameter -= learning_rate * (parameter.grad / scale)
            parameter.grad = None


def _ratio(numerator: int, denominator: int) -> float:
    """Zero when the denominator is zero."""
    if denominator == 0:
        return 0.0
    return float(numerator) / float(denominator)


def _f1(precision: float, recall: float) -> float:
    """Harmonic mean. Zero when both inputs are zero."""
    if precision + recall == 0.0:
        return 0.0
    return 2.0 * precision * recall / (precision + recall)


## Optuna search

In [ ]:
"""Purged Bayesian search for the recognition TCN.

The study never receives the sealed test ends. Its score is recognition
macro-F1, not a trading result.
"""

from __future__ import annotations

from typing import Any, Mapping, Sequence

import numpy as np
import optuna


optuna.logging.set_verbosity(optuna.logging.WARNING)


def run_optuna_search(
    prepared: Mapping[str, object],
    train_ends: np.ndarray,
    val_ends: np.ndarray,
    *,
    n_trials: int = SEARCH_TRIALS,
    max_epochs: int = SEARCH_EPOCHS,
    patience: int = SEARCH_PATIENCE,
    seed: int = SEED,
    hidden_choices: Sequence[int] = HIDDEN_CHOICES,
    batch_choices: Sequence[int] = BATCH_CHOICES,
    clip_choices: Sequence[float] = CLIP_CHOICES,
    search_stride: int = TRAIN_STRIDE,
    folds: int = DEVELOPMENT_FOLDS,
) -> dict[str, Any]:
    """Search TCN settings on development folds.

    Args:
        prepared: Feature frame and labels from ``prepare_series``.
        train_ends: Outer training ends. Test ends must not be included.
        val_ends: Outer validation ends. Test ends must not be included.
        n_trials: Optuna trials.
        max_epochs: Epoch cap inside each trial.
        patience: Early-stopping patience on fold validation loss.
        seed: Sampler seed.
        hidden_choices: Encoder widths.
        batch_choices: Mini-batch sizes.
        clip_choices: Positive-class weight caps.
        search_stride: Training-window stride used inside the study.
        folds: Chronological development folds.

    Returns:
        The best trial parameters and the mean recognition score.

    Raises:
        ValueError: If a fold overlaps a caller-supplied sealed test.
    """
    fold_rows = development_folds(train_ends, val_ends, folds=folds, purge=PURGE_BARS)
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=seed),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=0),
    )

    def objective(trial: optuna.Trial) -> float:
        depth = trial.suggest_int("dilation_depth", 2, 4)
        params = {
            "hidden": int(trial.suggest_categorical("hidden", list(hidden_choices))),
            "dilations": tuple(2**index for index in range(depth)),
            "learning_rate": float(
                trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
            ),
            "batch_size": int(trial.suggest_categorical("batch_size", list(batch_choices))),
            "pos_weight_clip": float(
                trial.suggest_categorical("pos_weight_clip", list(clip_choices))
            ),
        }
        scores: list[float] = []
        for step, fold in enumerate(fold_rows):
            scores.append(
                _fold_score(
                    prepared,
                    fold,
                    params,
                    max_epochs=max_epochs,
                    patience=patience,
                    seed=seed + trial.number,
                    search_stride=search_stride,
                )
            )
            trial.report(float(np.mean(scores)), step)
            if trial.should_prune():
                raise optuna.TrialPruned()
        return float(np.mean(scores)) if scores else 0.0

    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    best = study.best_trial
    depth = int(best.params["dilation_depth"])
    return {
        "best_value": float(best.value) if best.value is not None else 0.0,
        "trials": _trial_rows(study),
        "best_params": {
            "hidden": int(best.params["hidden"]),
            "dilation_depth": depth,
            "dilations": [int(2**index) for index in range(depth)],
            "learning_rate": float(best.params["learning_rate"]),
            "batch_size": int(best.params["batch_size"]),
            "pos_weight_clip": float(best.params["pos_weight_clip"]),
        },
        "n_trials": int(len(study.trials)),
        "folds": [
            {
                "train_windows": int(len(fold["train"])),
                "val_windows": int(len(fold["val"])),
                "purge_gap_bars": int(fold["val"][0] - fold["train"][-1]),
            }
            for fold in fold_rows
        ],
    }


def _trial_rows(study: optuna.Study) -> list[dict[str, Any]]:
    """One row per Optuna trial, including trials the pruner stopped."""
    rows: list[dict[str, Any]] = []
    for trial in study.trials:
        raw = trial.value
        value = None
        if raw is not None and np.isfinite(float(raw)):
            value = float(raw)
        rows.append(
            {
                "number": int(trial.number),
                "value": value,
                "state": str(trial.state.name),
            }
        )
    return rows


def assert_folds_exclude_test(
    folds: Sequence[Mapping[str, np.ndarray]],
    test_ends: np.ndarray,
) -> None:
    """Raise when a development fold contains a sealed test end."""
    sealed = {int(end) for end in np.asarray(test_ends, dtype=np.int64)}
    for fold in folds:
        seen = {int(end) for end in fold["train"]} | {int(end) for end in fold["val"]}
        leaked = sorted(sealed & seen)
        if leaked:
            raise ValueError(f"development fold contains sealed test ends: {leaked[:5]}")


def _fold_score(
    prepared: Mapping[str, object],
    fold: Mapping[str, np.ndarray],
    params: Mapping[str, Any],
    *,
    max_epochs: int,
    patience: int,
    seed: int,
    search_stride: int,
) -> float:
    """Recognition score for one purged fold. The test split is not read."""
    train_ends = strided_ends(fold["train"], search_stride)
    val_ends = nonoverlapping_ends(fold["val"], EVAL_STRIDE)
    if train_ends.size < 2 or val_ends.size < 1:
        return 0.0
    train, val = _scaled_pair(prepared, train_ends, val_ends)
    trained = train_tcn(
        train,
        val,
        None,
        epochs=max_epochs,
        batch_size=int(params["batch_size"]),
        seed=seed,
        hidden=int(params["hidden"]),
        dilations=tuple(int(item) for item in params["dilations"]),
        learning_rate=float(params["learning_rate"]),
        pos_weight_clip=float(params["pos_weight_clip"]),
        patience=patience,
    )
    geometry = float(trained["val"]["geometry_macro_f1"])
    state = float(trained["val"]["market_state"]["macro_f1"])
    return 0.5 * (geometry + state)


def _scaled_pair(
    prepared: Mapping[str, object],
    train_ends: np.ndarray,
    val_ends: np.ndarray,
) -> tuple[dict[str, np.ndarray], dict[str, np.ndarray]]:
    """Standardize fold windows with a scaler fit on that fold's train rows."""
    train = stack_windows(prepared, train_ends, "engineered")
    val = stack_windows(prepared, val_ends, "engineered")
    scaler = ChannelScaler().fit(train["features"])
    train["features"] = scaler.transform(train["features"])
    val["features"] = scaler.transform(val["features"])
    return train, val


## Correlation and TreeSHAP

In [ ]:
"""Train-only correlation and TreeSHAP for the recognition features.

Neither audit adds a column. Both refuse names outside the allow-list.
"""

from __future__ import annotations

from typing import Any, Sequence

import numpy as np
from scipy.stats import spearmanr
from xgboost import XGBClassifier



def spearman_report(
    features: np.ndarray,
    geometry: np.ndarray,
    state: np.ndarray,
    channels: Sequence[str] = ENGINEERED_CHANNELS,
    *,
    flag_at: float = CORRELATION_FLAG,
) -> dict[str, Any]:
    """Spearman correlation of allowed channels with recognition labels.

    Args:
        features: Training windows, shape ``(n, time, channels)``.
        geometry: Per-bar geometric labels aligned with ``features``.
        state: Window market-state codes.
        channels: Channel names. Must be the recognition allow-list.
        flag_at: Absolute correlation that is reported as high.

    Returns:
        Pair rows and the high-correlation subset. A high value means the
        geometry rule is visible in an allowed channel. It does not add a
        feature.
    """
    assert_feature_names(tuple(channels))
    last = features[:, -1, :]
    pooled = features.mean(axis=1)
    last_labels = geometry[:, -1, :]
    presence = (geometry.max(axis=1) >= 0.5).astype(np.float64)
    rows: list[dict[str, Any]] = []
    for channel_index, channel in enumerate(channels):
        for class_index, class_name in enumerate(GEOMETRIC_CLASSES):
            rows.append(
                _pair(
                    channel,
                    class_name,
                    "last_bar",
                    _spearman(last[:, channel_index], last_labels[:, class_index]),
                    flag_at,
                )
            )
            rows.append(
                _pair(
                    channel,
                    class_name,
                    "window_mean",
                    _spearman(pooled[:, channel_index], presence[:, class_index]),
                    flag_at,
                )
            )
        rows.append(
            _pair(
                channel,
                "market_state",
                "last_bar",
                _spearman(last[:, channel_index], state.astype(np.float64)),
                flag_at,
            )
        )
        rows.append(
            _pair(
                channel,
                "market_state",
                "window_mean",
                _spearman(pooled[:, channel_index], state.astype(np.float64)),
                flag_at,
            )
        )
    flagged = [row for row in rows if row["flagged"]]
    return {"threshold": float(flag_at), "pairs": rows, "flagged": flagged}


def presence_shap(
    features: np.ndarray,
    geometry: np.ndarray,
    channels: Sequence[str] = ENGINEERED_CHANNELS,
    *,
    sample_size: int = SHAP_SAMPLE,
    n_estimators: int = XGB_ESTIMATORS,
    seed: int = SEED,
) -> dict[str, Any]:
    """TreeSHAP of the presence baseline, summed back to input channels.

    Args:
        features: Development windows the trees are fit on.
        geometry: Geometric labels for those windows.
        channels: Allow-listed channel names.
        sample_size: Maximum windows passed to the explainer.
        n_estimators: Trees per family.
        seed: Sample and tree seed.

    Returns:
        Mean absolute SHAP by channel for each family that had both classes.
    """
    assert_feature_names(tuple(channels))
    import shap

    flat = features.reshape(len(features), -1)
    presence = (geometry.max(axis=1) >= 0.5).astype(np.int64)
    rng = np.random.default_rng(seed)
    take = min(int(sample_size), len(features))
    chosen = np.sort(rng.choice(len(features), size=take, replace=False))
    explain_rows = flat[chosen]
    per_class: dict[str, Any] = {}
    for class_index, name in enumerate(GEOMETRIC_CLASSES):
        labels = presence[:, class_index]
        positives = int(labels.sum())
        negatives = int(len(labels) - positives)
        if positives == 0 or negatives == 0:
            per_class[name] = {"skipped": True}
            continue
        model = XGBClassifier(
            n_estimators=n_estimators,
            max_depth=XGB_MAX_DEPTH,
            learning_rate=0.1,
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=float(negatives) / float(positives),
            tree_method="hist",
            n_jobs=1,
            random_state=seed,
        )
        model.fit(flat, labels)
        explainer = shap.TreeExplainer(model)
        values = explainer.shap_values(explain_rows)
        width = int(explain_rows.shape[1])
        summary = aggregate_shap_to_channels(
            _positive_shap(values, width),
            n_time=WINDOW_BARS,
            n_channels=len(channels),
        )
        per_class[name] = {
            "skipped": False,
            "channels": {
                channel: float(summary[index]) for index, channel in enumerate(channels)
            },
        }
    return {"sample_size": int(take), "per_class": per_class}


def aggregate_shap_to_channels(
    flat_importance: np.ndarray,
    *,
    n_time: int,
    n_channels: int,
) -> np.ndarray:
    """Sum a flattened window explanation back onto the channel axis.

    Args:
        flat_importance: Mean absolute SHAP with one value per flattened input.
        n_time: Bars in the window.
        n_channels: Channels per bar.

    Returns:
        Nonnegative importance per channel.
    """
    flat = np.asarray(flat_importance, dtype=np.float64).reshape(-1)
    expected = int(n_time) * int(n_channels)
    if flat.size != expected:
        raise ValueError(f"expected {expected} SHAP values, got {flat.size}")
    return np.abs(flat.reshape(int(n_time), int(n_channels))).sum(axis=0)


def _positive_shap(values: object, width: int) -> np.ndarray:
    """Reduce TreeSHAP output to one mean-absolute vector of ``width``."""
    if isinstance(values, list):
        values = values[-1]
    array = np.asarray(values, dtype=np.float64)
    if array.ndim == 1 and array.size == width:
        return np.abs(array)
    if array.shape[-1] == width:
        axes = tuple(range(array.ndim - 1))
        return np.mean(np.abs(array), axis=axes)
    raise ValueError(f"cannot align SHAP shape {array.shape} to width {width}")


def _pair(
    channel: str,
    label: str,
    reduction: str,
    coefficient: float,
    flag_at: float,
) -> dict[str, Any]:
    """One correlation row."""
    return {
        "channel": channel,
        "label": label,
        "reduction": reduction,
        "spearman": float(coefficient),
        "flagged": bool(abs(coefficient) >= flag_at),
    }


def _spearman(left: np.ndarray, right: np.ndarray) -> float:
    """Spearman correlation, or zero when either side has no variation."""
    mask = np.isfinite(left) & np.isfinite(right)
    if int(mask.sum()) < 8:
        return 0.0
    lhs = np.asarray(left[mask], dtype=np.float64)
    rhs = np.asarray(right[mask], dtype=np.float64)
    if float(np.std(lhs)) < 1e-12 or float(np.std(rhs)) < 1e-12:
        return 0.0
    result = spearmanr(lhs, rhs)
    coefficient = getattr(result, "statistic", None)
    if coefficient is None:
        coefficient = result[0]
    if coefficient is None or not np.isfinite(float(coefficient)):
        return 0.0
    return float(coefficient)


## Figures

In [ ]:
"""Figures for the recognition report.

The drawings read the report dict. They do not refit a model and they do
not add a feature.
"""

from __future__ import annotations

from pathlib import Path
from typing import Any, Mapping, Sequence

import numpy as np

Figure = Any


def write_recognition_figures(report: Mapping[str, Any], directory: Path) -> list[Path]:
    """Save one PNG per available section of ``report``.

    Args:
        report: Recognition report produced by ``run_colab``.
        directory: Folder that already holds the checkpoint.

    Returns:
        Paths of the figures that had enough data to draw.
    """
    directory.mkdir(parents=True, exist_ok=True)
    written: list[Path] = []
    for name, figure in recognition_figures(report):
        path = directory / f"{name}.png"
        figure.savefig(path, dpi=120, bbox_inches="tight")
        _close(figure)
        written.append(path)
    return written


def show_recognition_plots(report: Mapping[str, Any]) -> None:
    """Draw the report figures in the current notebook output."""
    import matplotlib.pyplot as plt

    figures = [figure for _name, figure in recognition_figures(report)]
    if not figures:
        print("report has no rows to plot")
        return
    plt.show()
    for figure in figures:
        _close(figure)


def recognition_figures(report: Mapping[str, Any]) -> list[tuple[str, Figure]]:
    """Build the figures that ``report`` can support.

    Args:
        report: In-memory recognition report.

    Returns:
        ``(file stem, figure)`` pairs. The caller closes each figure.
    """
    import matplotlib.pyplot as plt

    plt.ioff()
    figures: list[tuple[str, Figure]] = []
    learning = _learning_curve(report)
    if learning is not None:
        figures.append(("learning_curve", learning))
    scores = _split_scores(report)
    if scores is not None:
        figures.append(("validation_and_test", scores))
    search = _optuna_trials(report)
    if search is not None:
        figures.append(("optuna_trials", search))
    correlation = _correlation_heatmaps(report)
    if correlation is not None:
        figures.append(("label_correlation", correlation))
    explained = _shap_heatmap(report)
    if explained is not None:
        figures.append(("shap_channels", explained))
    reliability = _calibration(report)
    if reliability is not None:
        figures.append(("calibration", reliability))
    return figures


def _learning_curve(report: Mapping[str, Any]) -> Figure | None:
    """Train and validation loss for the refit that is saved."""
    import matplotlib.pyplot as plt

    fit = report.get("fit")
    if not isinstance(fit, Mapping):
        return None
    train = [float(item) for item in fit.get("train_loss_by_epoch", [])]
    val = [float(item) for item in fit.get("val_loss_by_epoch", [])]
    if not val:
        return None
    epochs = np.arange(1, len(val) + 1)
    figure, axis = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
    if len(train) == len(val):
        axis.plot(epochs, train, marker="o", label="train")
    axis.plot(epochs, val, marker="o", label="validation")
    best = int(fit.get("best_epoch", int(np.argmin(val)))) + 1
    axis.axvline(best, color="0.4", linestyle="--", label=f"kept epoch {best}")
    axis.set_xlabel("epoch")
    axis.set_ylabel("loss")
    axis.set_title("Final TCN: train loss and validation loss")
    axis.legend()
    return figure


def _split_scores(report: Mapping[str, Any]) -> Figure | None:
    """Macro-F1 and per-class geometry F1 on validation and the sealed test."""
    import matplotlib.pyplot as plt

    val = report.get("val")
    test = report.get("test")
    if not isinstance(val, Mapping) or not isinstance(test, Mapping):
        return None
    labels, val_macro, test_macro = _macro_trio(val, test)
    if not labels:
        return None
    figure, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
    positions = np.arange(len(labels))
    width = 0.36
    axes[0].bar(positions - width / 2, val_macro, width, label="validation")
    axes[0].bar(positions + width / 2, test_macro, width, label="sealed test")
    axes[0].set_xticks(positions, labels, rotation=15, ha="right")
    axes[0].set_ylim(0.0, 1.0)
    axes[0].set_ylabel("macro-F1")
    axes[0].set_title("Chosen epoch")
    axes[0].legend()
    _grouped_class_bars(
        axes[1],
        _class_f1(val.get("geometry_per_class")),
        _class_f1(test.get("geometry_per_class")),
    )
    axes[1].set_ylim(0.0, 1.0)
    axes[1].set_ylabel("F1")
    axes[1].set_title("Geometry families")
    axes[1].legend()
    figure.suptitle("Validation against the sealed test, scored once")
    return figure


def _optuna_trials(report: Mapping[str, Any]) -> Figure | None:
    """Trial scores from the purged search. Pruned trials stay visible."""
    import matplotlib.pyplot as plt

    study = report.get("optuna")
    if not isinstance(study, Mapping):
        return None
    trials = [row for row in study.get("trials", []) if isinstance(row, Mapping)]
    if not trials:
        return None
    figure, axis = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
    best_so_far: list[float] = []
    running: float | None = None
    missing: list[int] = []
    for row in trials:
        number = int(row["number"])
        value = row.get("value")
        completed = str(row.get("state")) == "COMPLETE" and value is not None
        if value is None:
            missing.append(number)
        else:
            color = "C0" if completed else "C1"
            axis.scatter(number, float(value), color=color, zorder=3)
        if completed:
            score = float(value)
            running = score if running is None else max(running, score)
        best_so_far.append(np.nan if running is None else running)
    axis.plot(
        [int(row["number"]) for row in trials],
        best_so_far,
        color="0.2",
        label="best completed",
    )
    if missing:
        floor = axis.get_ylim()[0]
        axis.scatter(missing, [floor] * len(missing), marker="x", color="0.5", zorder=3)
    axis.scatter([], [], color="C0", label="completed")
    axis.scatter([], [], color="C1", label="pruned with a score")
    axis.scatter([], [], marker="x", color="0.5", label="pruned before a score")
    axis.set_xlabel("trial")
    axis.set_ylabel("mean macro-F1")
    axis.set_title("Development search (validation folds only)")
    axis.legend()
    return figure


def _correlation_heatmaps(report: Mapping[str, Any]) -> Figure | None:
    """Spearman of each channel with each recognition label."""
    import matplotlib.pyplot as plt

    block = report.get("correlation")
    if not isinstance(block, Mapping):
        return None
    pairs = [row for row in block.get("pairs", []) if isinstance(row, Mapping)]
    if not pairs:
        return None
    threshold = float(block.get("threshold", 0.5))
    figure, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
    image = None
    for axis, reduction in zip(axes, ("last_bar", "window_mean")):
        channels, labels, grid = _pivot_correlation(pairs, reduction)
        if grid.size == 0:
            axis.set_axis_off()
            continue
        image = axis.imshow(grid, cmap="RdBu_r", vmin=-1.0, vmax=1.0, aspect="auto")
        axis.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
        axis.set_yticks(range(len(channels)), channels)
        axis.set_title(reduction.replace("_", " "))
        _annotate_grid(axis, grid, threshold)
    if image is None:
        _close(figure)
        return None
    figure.colorbar(image, ax=axes, fraction=0.03, label="Spearman")
    figure.suptitle(
        f"Train windows only. A black box marks |Spearman| >= {threshold:.1f}"
    )
    return figure


def _shap_heatmap(report: Mapping[str, Any]) -> Figure | None:
    """Channel importance from TreeSHAP, one column per geometric family."""
    import matplotlib.pyplot as plt

    block = report.get("shap")
    if not isinstance(block, Mapping):
        return None
    per_class = block.get("per_class")
    if not isinstance(per_class, Mapping):
        return None
    classes: list[str] = []
    channels: list[str] = []
    rows: list[list[float]] = []
    for name, item in per_class.items():
        if not isinstance(item, Mapping) or item.get("skipped"):
            continue
        scores = item.get("channels")
        if not isinstance(scores, Mapping):
            continue
        if not channels:
            channels = [str(channel) for channel in scores]
        classes.append(str(name))
        rows.append([float(scores.get(channel, 0.0)) for channel in channels])
    if not rows:
        return None
    grid = np.asarray(rows, dtype=np.float64).T
    order = np.argsort(grid.mean(axis=1))[::-1]
    grid = grid[order]
    channels = [channels[int(index)] for index in order]
    figure, axes = plt.subplots(
        1,
        2,
        figsize=(14, 6),
        constrained_layout=True,
        gridspec_kw={"width_ratios": [2.2, 1.0]},
    )
    image = axes[0].imshow(grid, cmap="YlOrRd", aspect="auto")
    axes[0].set_xticks(range(len(classes)), classes, rotation=45, ha="right")
    axes[0].set_yticks(range(len(channels)), channels)
    axes[0].set_title("By family")
    figure.colorbar(image, ax=axes[0], fraction=0.05, label="mean |SHAP|")
    means = grid.mean(axis=1)
    positions = np.arange(len(channels))
    axes[1].barh(positions, means)
    axes[1].set_yticks(positions, channels)
    axes[1].invert_yaxis()
    axes[1].set_xlabel("mean |SHAP|")
    axes[1].set_title("Across families")
    sample = block.get("sample_size")
    figure.suptitle(f"TreeSHAP on the presence baseline (n={sample})")
    return figure


def _calibration(report: Mapping[str, Any]) -> Figure | None:
    """Reliability of Head A probabilities on the sealed test."""
    import matplotlib.pyplot as plt

    test = report.get("test")
    if not isinstance(test, Mapping):
        return None
    table = test.get("geometry_calibration")
    if not isinstance(table, Mapping):
        return None
    bins = [row for row in table.get("bins", []) if isinstance(row, Mapping)]
    if not bins:
        return None
    confidence = [float(row["confidence"]) for row in bins]
    frequency = [float(row["frequency"]) for row in bins]
    counts = np.asarray([float(row["count"]) for row in bins], dtype=np.float64)
    scale = 40.0 + 400.0 * counts / max(float(counts.max()), 1.0)
    figure, axis = plt.subplots(figsize=(6, 5), constrained_layout=True)
    axis.plot([0.0, 1.0], [0.0, 1.0], linestyle="--", color="0.5", label="matches")
    axis.scatter(confidence, frequency, s=scale, label="bin")
    axis.set_xlim(0.0, 1.0)
    axis.set_ylim(0.0, 1.0)
    axis.set_xlabel("predicted probability")
    axis.set_ylabel("observed frequency")
    error = float(table.get("expected_calibration_error", 0.0))
    axis.set_title(f"Head A reliability on the sealed test (ECE {error:.3f})")
    axis.legend()
    return figure


def _macro_trio(
    val: Mapping[str, Any],
    test: Mapping[str, Any],
) -> tuple[list[str], list[float], list[float]]:
    """Macro-F1 labels shared by both splits."""
    labels: list[str] = []
    left: list[float] = []
    right: list[float] = []
    pairs = (
        ("geometry", "geometry_macro_f1", None),
        ("market state", "market_state", "macro_f1"),
        ("structure tag", "structure_tag", "macro_f1"),
    )
    for label, key, nested in pairs:
        val_score = _nested_float(val, key, nested)
        test_score = _nested_float(test, key, nested)
        if val_score is None or test_score is None:
            continue
        labels.append(label)
        left.append(val_score)
        right.append(test_score)
    return labels, left, right


def _nested_float(
    block: Mapping[str, Any],
    key: str,
    nested: str | None,
) -> float | None:
    """Read a float, or a float inside one nested mapping."""
    value = block.get(key)
    if nested is None:
        if isinstance(value, (int, float)):
            return float(value)
        return None
    if isinstance(value, Mapping) and isinstance(value.get(nested), (int, float)):
        return float(value[nested])
    return None


def _class_f1(per_class: object) -> list[tuple[str, float]]:
    """F1 for each class that published a score."""
    if not isinstance(per_class, Mapping):
        return []
    rows: list[tuple[str, float]] = []
    for name, scores in per_class.items():
        if isinstance(scores, Mapping) and isinstance(scores.get("f1"), (int, float)):
            rows.append((str(name), float(scores["f1"])))
    return rows


def _grouped_class_bars(
    axis: Any,
    val_rows: Sequence[tuple[str, float]],
    test_rows: Sequence[tuple[str, float]],
) -> None:
    """Side-by-side F1 bars. Classes missing from one split are skipped."""
    test_map = dict(test_rows)
    names = [name for name, _score in val_rows if name in test_map]
    if not names:
        axis.set_axis_off()
        return
    val_map = dict(val_rows)
    positions = np.arange(len(names))
    width = 0.36
    axis.bar(
        positions - width / 2,
        [val_map[name] for name in names],
        width,
        label="validation",
    )
    axis.bar(
        positions + width / 2,
        [test_map[name] for name in names],
        width,
        label="sealed test",
    )
    axis.set_xticks(positions, names, rotation=45, ha="right")


def _pivot_correlation(
    pairs: Sequence[Mapping[str, Any]],
    reduction: str,
) -> tuple[list[str], list[str], np.ndarray]:
    """Channel by label Spearman grid for one reduction."""
    chosen = [row for row in pairs if row.get("reduction") == reduction]
    channels = list(dict.fromkeys(str(row["channel"]) for row in chosen))
    labels = list(dict.fromkeys(str(row["label"]) for row in chosen))
    grid = np.full((len(channels), len(labels)), np.nan, dtype=np.float64)
    channel_index = {name: index for index, name in enumerate(channels)}
    label_index = {name: index for index, name in enumerate(labels)}
    for row in chosen:
        grid[channel_index[str(row["channel"])], label_index[str(row["label"])]] = float(
            row["spearman"]
        )
    return channels, labels, grid


def _annotate_grid(axis: Any, grid: np.ndarray, threshold: float) -> None:
    """Print each correlation and box the pairs past the flag."""
    from matplotlib.patches import Rectangle

    for row in range(grid.shape[0]):
        for col in range(grid.shape[1]):
            value = float(grid[row, col])
            if not np.isfinite(value):
                continue
            axis.text(
                col,
                row,
                f"{value:.2f}",
                ha="center",
                va="center",
                fontsize=7,
                color="white" if abs(value) >= 0.65 else "black",
            )
            if abs(value) >= threshold:
                axis.add_patch(
                    Rectangle(
                        (col - 0.5, row - 0.5),
                        1,
                        1,
                        fill=False,
                        edgecolor="black",
                        linewidth=1.2,
                    )
                )


def _close(figure: Figure) -> None:
    """Release a figure without drawing it again."""
    import matplotlib.pyplot as plt

    plt.close(figure)


## Colab entry point

In [ ]:
"""Colab entry point for the intense recognition run.

The sealed test is scored once. The report is not an order and it is not
copied into the live model directory.
"""

from __future__ import annotations

import json
import zipfile
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch

_FORBIDDEN_REPORT_TEXT = (
    "expectancy",
    "sharpe",
    "model_storage",
    "forecastmodelnode",
    "orders_enabled",
    "buy_signal",
    "sell_signal",
)

def run_colab(
    source: Path,
    output_dir: Path | None = None,
    *,
    n_trials: int = SEARCH_TRIALS,
    seed: int = SEED,
    shap_sample: int = SHAP_SAMPLE,
) -> dict[str, Any]:
    """Search, explain, refit, and score the sealed recognition test.

    Args:
        source: 5-minute BTCUSD parquet.
        output_dir: Directory for the report, checkpoint, and zip.
        n_trials: Optuna trials on the development span.
        seed: Split and sampler seed.
        shap_sample: Development windows explained by TreeSHAP.

    Returns:
        The recognition report. It has no order fields.
    """
    destination = output_dir or (_repo_root() / COLAB_OUTPUT_RELATIVE).parent
    destination.mkdir(parents=True, exist_ok=True)
    raw = pd.read_parquet(source)
    bars = resample_completed_15m(dedupe_timestamps(raw))
    prepared = prepare_series(bars)
    if not rule_card_is_separate(prepared):
        raise ValueError("rule card was not separated from the feature tensor")
    prepared.pop("rule_card")
    splits = prepared["splits"]
    if not isinstance(splits, dict):
        raise TypeError("prepared series is missing splits")
    folds = development_folds(splits["train"], splits["val"])
    assert_folds_exclude_test(folds, splits["test"])
    indices = perturbation_indices(len(bars), first_finite_bar(prepared["features"]))
    for index in indices:
        assert_causal_at(bars, index)
    study = run_optuna_search(
        prepared,
        splits["train"],
        splits["val"],
        n_trials=n_trials,
        seed=seed,
    )
    train_view = strided_ends(splits["train"], TRAIN_STRIDE)
    audit = stack_windows(prepared, train_view, "engineered")
    correlation = spearman_report(
        audit["features"],
        audit["geometry"],
        audit["state"],
        tuple(audit["channels"]),
    )
    explained = presence_shap(
        audit["features"],
        audit["geometry"],
        tuple(audit["channels"]),
        sample_size=shap_sample,
        seed=seed,
    )
    stride = final_train_stride(int(len(splits["train"])), len(ENGINEERED_CHANNELS))
    trained = _refit(
        prepared,
        splits,
        study["best_params"],
        stride,
        destination / "tcn_colab.pt",
        seed,
    )
    report = {
        "spec_version": SPEC_VERSION,
        "rows_5m": int(len(raw)),
        "rows_15m": int(len(bars)),
        "heads_not_trained": list(HEADS_NOT_TRAINED),
        "perturbation_indices": indices,
        "purge_gap_bars": {
            "train_val": int(splits["val"][0] - splits["train"][-1]),
            "val_test": int(splits["test"][0] - splits["val"][-1]),
        },
        "optuna": study,
        "correlation": {
            "threshold": correlation["threshold"],
            "pairs": correlation["pairs"],
            "flagged": correlation["flagged"],
            "pair_count": len(correlation["pairs"]),
        },
        "shap": explained,
        "final_stride_bars": stride,
        "eval_stride_bars": EVAL_STRIDE,
        "optimizer": trained["optimizer"],
        "fit": {
            "best_epoch": int(trained["best_epoch"]),
            "train_loss_by_epoch": [float(item) for item in trained["train_loss_by_epoch"]],
            "val_loss_by_epoch": [float(item) for item in trained["val_loss_by_epoch"]],
        },
        "val": trained["val"],
        "test": trained["test"],
        "sealed_test_scored_once": True,
    }
    figure_paths = write_recognition_figures(report, destination)
    report["figures"] = [path.name for path in figure_paths]
    assert_recognition_report(report)
    report_path = destination / "report.json"
    report_path.write_text(json.dumps(_json_ready(report), indent=2), encoding="utf-8")
    archive = destination / "recognition_15m_colab.zip"
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as handle:
        handle.write(report_path, arcname="report.json")
        handle.write(destination / "tcn_colab.pt", arcname="tcn_colab.pt")
        for figure_path in figure_paths:
            handle.write(figure_path, arcname=figure_path.name)
    report["report_path"] = str(report_path)
    report["zip_path"] = str(archive)
    return report

def perturbation_indices(n_bars: int, first_finite: int) -> list[int]:
    """Three bar indices with later bars available for a causality check.

    Args:
        n_bars: Length of the 15-minute series.
        first_finite: First bar whose engineered features are finite.

    Returns:
        Start, middle, and late indices. Each has at least one bar after it.

    Raises:
        ValueError: If the series cannot support the check.
    """
    start = first_finite + WINDOW_BARS
    late = n_bars - 2
    if start >= late:
        raise ValueError("series is too short for a future-bar check")
    mid = (start + late) // 2
    return [int(start), int(mid), int(late)]

def final_train_stride(n_train_ends: int, n_channels: int) -> int:
    """Use every training window on a GPU when the tensor fits.

    Args:
        n_train_ends: Windows in the outer training split.
        n_channels: Engineered channel count.

    Returns:
        1 or 2 on a GPU with room, otherwise the search stride of 4.
    """
    if not torch.cuda.is_available():
        return TRAIN_STRIDE
    estimate = int(n_train_ends) * WINDOW_BARS * int(n_channels) * 4
    if estimate <= 1_500_000_000:
        return 1
    if estimate <= 3_000_000_000:
        return 2
    return TRAIN_STRIDE

def _repo_root() -> Path:
    """Repository root when this file is on disk, otherwise the working directory.

    A Colab cell has no ``__file__``. The notebook passes an output directory,
    and this fallback keeps a local call working from the repo root.
    """
    located = globals().get("__file__")
    if isinstance(located, str) and located:
        return Path(located).resolve().parents[2]
    return Path.cwd()

def _json_ready(value: Any) -> Any:
    """Convert numpy values so the report file is plain JSON."""
    if isinstance(value, dict):
        return {str(key): _json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_ready(item) for item in value]
    if isinstance(value, np.ndarray):
        return _json_ready(value.tolist())
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.integer):
        return int(value)
    return value

def assert_recognition_report(report: dict[str, Any]) -> None:
    """Reject a report that carries an order or promotion field.

    Args:
        report: JSON-ready recognition report.

    Raises:
        ValueError: If forbidden trading text appears in the serialized report.
    """
    blob = json.dumps(_json_ready(report)).lower()
    found = [token for token in _FORBIDDEN_REPORT_TEXT if token in blob]
    if found:
        raise ValueError(f"recognition report contains trading fields: {found}")

def _refit(
    prepared: dict[str, object],
    splits: dict[str, np.ndarray],
    params: dict[str, Any],
    stride: int,
    checkpoint: Path,
    seed: int,
) -> dict[str, Any]:
    """Refit the chosen TCN and score the sealed test once."""
    train_ends = strided_ends(splits["train"], stride)
    val_ends = nonoverlapping_ends(splits["val"], EVAL_STRIDE)
    test_ends = nonoverlapping_ends(splits["test"], EVAL_STRIDE)
    train = stack_windows(prepared, train_ends, "engineered")
    val = stack_windows(prepared, val_ends, "engineered")
    test = stack_windows(prepared, test_ends, "engineered")
    scaler = ChannelScaler().fit(train["features"])
    for batch in (train, val, test):
        batch["features"] = scaler.transform(batch["features"])
    trained = train_tcn(
        train,
        val,
        test,
        epochs=FINAL_EPOCHS,
        batch_size=int(params["batch_size"]),
        seed=seed,
        hidden=int(params["hidden"]),
        dilations=tuple(int(item) for item in params["dilations"]),
        learning_rate=float(params["learning_rate"]),
        pos_weight_clip=float(params["pos_weight_clip"]),
        patience=FINAL_PATIENCE,
        include_test_calibration=True,
    )
    state = trained.pop("state_dict")
    checkpoint.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "spec_version": SPEC_VERSION,
            "state_dict": state,
            "best_params": params,
            "scaler_mean": scaler.mean,
            "scaler_std": scaler.std,
            "channels": list(ENGINEERED_CHANNELS),
        },
        checkpoint,
    )
    return trained


## Fetch the 5-minute candles

In [ ]:
import json
import time
import urllib.parse
import urllib.request
from pathlib import Path

import pandas as pd

# Public history. No API key. One response holds about 4,000 candles.
DELTA_CANDLES_URL = "https://api.india.delta.exchange/v2/history/candles"
PAGE_BARS = 4_000
BAR_SECONDS = 300
TARGET_BARS = 100_000
MIN_BARS = 15_000
LOOKBACK_DAYS = 730
PAUSE_SECONDS = 0.35


def _candle_page(start: int, end: int) -> list[dict]:
    """Request one public BTCUSD 5-minute page."""
    query = urllib.parse.urlencode(
        {
            "symbol": "BTCUSD",
            "resolution": "5m",
            "start": int(start),
            "end": int(end),
        }
    )
    request = urllib.request.Request(
        f"{DELTA_CANDLES_URL}?{query}",
        headers={
            "Accept": "application/json",
            "User-Agent": "JackSparrow-recognition/1.0",
        },
    )
    with urllib.request.urlopen(request, timeout=60) as response:
        payload = json.loads(response.read().decode("utf-8"))
    if payload.get("success") is False:
        raise RuntimeError("Delta candle history returned success=false")
    result = payload.get("result", [])
    if isinstance(result, dict):
        result = result.get("candles", [])
    if not isinstance(result, list):
        raise RuntimeError("Delta candle history did not return a list")
    return result


def fetch_btcusd_5m(target: int = TARGET_BARS) -> pd.DataFrame:
    """Walk backward through public 5-minute candles and return OHLCV."""
    end = int(time.time())
    earliest = end - LOOKBACK_DAYS * 24 * 60 * 60
    rows: list[dict] = []
    seen: set[int] = set()
    while len(rows) < target and end > earliest:
        start = max(earliest, end - PAGE_BARS * BAR_SECONDS)
        page = _candle_page(start, end)
        if not page:
            break
        oldest = end
        added = 0
        for row in page:
            stamp = int(row["time"])
            oldest = min(oldest, stamp)
            if stamp in seen:
                continue
            seen.add(stamp)
            rows.append(row)
            added += 1
        nxt = oldest - 1
        if nxt >= end or added == 0:
            break
        end = nxt
        print(f"delta 5m candles: {len(rows)}", flush=True)
        if len(rows) >= target:
            break
        time.sleep(PAUSE_SECONDS)
    if len(rows) < MIN_BARS:
        raise RuntimeError(
            f"Delta returned {len(rows)} five-minute candles; "
            f"need at least {MIN_BARS}."
        )
    frame = pd.DataFrame(rows)
    frame["time"] = pd.to_datetime(frame["time"].astype("int64"), unit="s", utc=True)
    for column in ("open", "high", "low", "close", "volume"):
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    frame = frame.dropna(subset=["time", "open", "high", "low", "close", "volume"])
    frame = frame.sort_values("time").drop_duplicates("time", keep="last")
    frame = frame.tail(target).reset_index(drop=True)
    return frame.loc[:, ["time", "open", "high", "low", "close", "volume"]]


parquet = Path("/content/btcusd_5m_raw.parquet")
candles = fetch_btcusd_5m()
candles.to_parquet(parquet, index=False)
print(
    parquet,
    len(candles),
    candles["time"].iloc[0],
    candles["time"].iloc[-1],
)


## Search, explain, and score the sealed test

In [ ]:
from pathlib import Path

report = run_colab(
    source=parquet,
    output_dir=Path("/content/export/recognition_15m/colab"),
)
print(report["test"]["geometry_macro_f1"])
print(report["test"]["geometry_calibration"]["expected_calibration_error"])
print(report["zip_path"])


## How to read the figures

The next cell draws the figures from the report already in memory. It does
not train again.

- **Learning curve.** Train loss and validation loss for the final TCN. The
  dashed line is the epoch whose weights are saved. A train line that keeps
  falling while validation turns up means later epochs fit the training
  windows more tightly than the validation windows.
- **Validation against the sealed test.** Macro-F1 for geometry, market
  state, and the structure tag, then F1 for each geometric family. The test
  bars are the one sealed score.
- **Development search.** Each dot is one Optuna trial. The line is the best
  completed trial so far. Pruned trials were stopped early on a validation fold.
- **Correlation.** Spearman on training windows only. The left heatmap uses
  the last bar of each window. The right heatmap uses the window mean. A black
  box marks a pair whose absolute correlation is at least 0.5. A high value
  on `body_to_range` versus `doji` is the candle rule showing through an
  allowed channel.
- **TreeSHAP.** Mean absolute contribution of each channel to the presence
  baseline, summed across the 48 bars. Taller bars are the channels the trees
  used. This ranking does not change the TCN.
- **Reliability.** For Head A on the sealed test, each point is a probability
  bin. A point on the dashed line means the predicted probability matches how
  often the family was present. Larger points are bins with more rows.


In [ ]:
show_recognition_plots(report)


## Download the research zip

In [ ]:
try:
    from google.colab import files
    files.download(report["zip_path"])
except Exception as exc:
    print("browser download skipped:", exc)
    print(report["zip_path"])
